
# Step 1b: Grad‑CAM Inspection Notebook

Use this notebook to **visualize what your classifier is focusing on** (e.g., human vs. avatar vs. animal).  
It supports:
- Loading a SavedModel (`.keras` or SavedModel dir) or a compiled `tf.keras` model
- Directory or CSV dataset indexing
- Generating Grad‑CAM heatmaps for **misclassified** and **correct** examples per class
- Saving side‑by‑side overlays to disk for reports
- (Optional) A simple **border‑attention score** that can hint at shortcut risks (logos/borders)



## 0) Requirements

```bash
pip install tensorflow pillow opencv-python-headless numpy pandas matplotlib tqdm scikit-learn
```



## 1) User Tunables


In [21]:
!pip install tensorflow pillow opencv-python-headless numpy pandas matplotlib tqdm scikit-learn
from pathlib import Path

# ===== USER TUNABLES =====
# MODEL_PATH = Path("models/baseline_savedmodel/resnet50_profilepic_classifier.keras")  # dir or file (.keras / .h5 / SavedModel dir)
MODEL_PATH = Path("models/resnet50_profilepic_classifier.keras")
CLASS_NAMES = None

DATA_ROOT = Path("final")
METADATA_CSV = None

TARGET_SPLIT = "val"
IMG_SIZE = 224
BATCH = 32

PREPROCESS = "resnet50"  # 'resnet50' | 'efficientnet' | 'none'
TARGET_LAYER_NAME = None # e.g., "conv5_block3_out"; None => auto-detect last conv
ALPHA = 0.35

N_MISCLASS_PER_CLASS = 8
N_CORRECT_PER_CLASS = 6

OUT_DIR = Path("gradcam_outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)
# =========================


## 2) Imports & GPU Check


In [22]:
import os, json, math, random
import numpy as np
import pandas as pd
import tensorflow as tf
try:
    from tensorflow.keras.applications import resnet50, efficientnet
except Exception:
    print('Using keras model instead of tensorflow.')
    from keras.applications import resnet50, efficientnet  # Keras 3 fallback
from typing import List, Tuple
from tqdm import tqdm

from PIL import Image
import matplotlib.pyplot as plt
import cv2

print("TF version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))


TF version: 2.20.0
GPU available: []



## 3) Load Model


In [23]:
def load_model_any(path: Path) -> tf.keras.Model:
    """Loads a TensorFlow model from various formats (.keras, .h5, SavedModel dir)."""
    if not path.exists():
        raise FileNotFoundError(f"Model path not found: {path}")

    try:
        # Check if the path is a directory (likely a SavedModel)
        if path.is_dir() and (path / 'saved_model.pb').exists():
            print(f"Loading SavedModel from directory: {path}")
            # Use TFSMLayer for SavedModel format
            # Assuming the default serving signature
            m = tf.keras.layers.TFSMLayer(str(path), call_endpoint='serving_default')
        else:
            print(f"Attempting to load model file: {path}")
            # Try loading as .keras or .h5
            m = tf.keras.models.load_model(path, compile=False)
            try:
                m.compile()
            except Exception:
                pass # Model might be loaded for inference only

        print("Model loaded successfully using TFSMLayer or load_model.")
        return m

    except Exception as e:
        raise RuntimeError(f"Error loading model from {path}: {e}")

print(f"Model Path {MODEL_PATH}")
print("Loading model from:", MODEL_PATH.resolve())
model = load_model_any(MODEL_PATH)
# After you load the model:
INPUT_KEY = model.inputs[0].name.split(":")[0]  # e.g., "image"
print("INPUT_KEY =", INPUT_KEY)

# TFSMLayer does not have a summary method like a standard Keras Model
# You might inspect the layer's inputs/outputs if needed
model.summary() # Commenting out as TFSMLayer doesn't have summary()

print("inputs:", [t.name for t in model.inputs])
print("shapes:", [t.shape for t in model.inputs])
print("dtypes:", [t.dtype for t in model.inputs])


Model Path models/resnet50_profilepic_classifier.keras
Loading model from: /content/models/resnet50_profilepic_classifier.keras
Attempting to load model file: models/resnet50_profilepic_classifier.keras
Model loaded successfully using TFSMLayer or load_model.
INPUT_KEY = image


Model: "resnet50_profilepic_classifier"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ image (InputLayer)  │ (None, 224, 224,  │          0 │ -                 │
│                     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ aug (Sequential)    │ (None, 224, 224,  │          0 │ image[0][0]       │
│                     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item_21         │ (None, 224, 224)  │          0 │ aug[0][0]         │
│ (GetItem)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item_22         │ (None, 224, 224)  │          0 │ aug[0][0]         │
│ (GetItem)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item_23         │ (None, 224, 224)  │          0 │ aug[0][0]         │
│ (GetItem)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stack_5 (Stack)     │ (None, 224, 224,  │          0 │ get_item_21[0][0… │
│                     │ 3)                │            │ get_item_22[0][0… │
│                     │                   │            │ get_item_23[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_7 (Add)         │ (None, 224, 224,  │          0 │ stack_5[0][0]     │
│                     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ resnet50            │ (None, 7, 7,      │ 23,587,712 │ add_7[0][0]       │
│ (Functional)        │ 2048)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 2048)      │          0 │ resnet50[0][0]    │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_18          │ (None, 2048)      │          0 │ global_average_p… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_18 (Dense)    │ (None, 256)       │    524,544 │ dropout_18[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_19          │ (None, 256)       │          0 │ dense_18[0][0]    │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_19 (Dense)    │ (None, 3)         │        771 │ dropout_19[0][0]  │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 24,113,027 (91.98 MB)

 Trainable params: 525,315 (2.00 MB)

 Non-trainable params: 23,587,712 (89.98 MB)

inputs: ['image']
shapes: [(None, 224, 224, 3)]
dtypes: ['float32']



## 4) Build Dataset Index (Directory or CSV)


In [24]:
def list_images_directory(root: Path, split: str) -> pd.DataFrame:
    print(f"Checking directory: {root / split}") # Added print statement
    rows = []
    base = root / split
    if not base.exists():
        print(f"Directory does not exist: {base}") # Added print statement
        return pd.DataFrame(columns=["path","label","split"])
    for cls_dir in sorted([p for p in base.iterdir() if p.is_dir()]):
        for img in cls_dir.rglob("*"):
            if img.suffix.lower() in {".jpg",".jpeg",".png",".bmp",".webp"}:
                rows.append({"path": str(img.as_posix()), "label": cls_dir.name, "split": split})
    return pd.DataFrame(rows)

if METADATA_CSV:
    df_all = pd.read_csv(METADATA_CSV)
    need = {"path","label","split"}
    if not need.issubset(set(df_all.columns)):
        raise ValueError(f"CSV must contain columns: {need}")
    df_all["path"] = df_all["path"].astype(str)
else:
    df_train = list_images_directory(DATA_ROOT, "train")
    df_val   = list_images_directory(DATA_ROOT, "val")
    df_test  = list_images_directory(DATA_ROOT, "test")
    df_all = pd.concat([df_train, df_val, df_test], ignore_index=True)
    print(f"head: {df_all.head()}")
    print(f"shape:{df_all.shape}")

df = df_all[df_all["split"] == TARGET_SPLIT].copy().reset_index(drop=True)

if df.empty:
    raise RuntimeError(f"Failed to build dataset index for split '{TARGET_SPLIT}'. No data found.")

if CLASS_NAMES is None:
    CLASS_NAMES = sorted(df_all["label"].dropna().unique().tolist())

label_to_index = {c:i for i,c in enumerate(CLASS_NAMES)}
index_to_label = {i:c for c,i in label_to_index.items()}

print("Classes:", CLASS_NAMES)
print("Counts:", df["label"].value_counts())

Checking directory: final/train
Checking directory: final/val
Checking directory: final/test
head:                                       path   label  split
0  final/train/animal/93ec4b691675b6c0.jpg  animal  train
1  final/train/animal/94421ccfa93847d3.jpg  animal  train
2  final/train/animal/d8be689340c99353.jpg  animal  train
3  final/train/animal/a06412878921e7c6.jpg  animal  train
4  final/train/animal/fba40c566603c69b.jpg  animal  train
shape:(29872, 3)
Classes: ['animal', 'avatar', 'human']
Counts: label
animal    1000
avatar    1000
human      987
Name: count, dtype: int64



## 5) tf.data Pipeline & Preprocessing


In [25]:
# from tensorflow.keras.applications import resnet50, efficientnet

def preprocess_image(path: tf.Tensor) -> tf.Tensor:
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE), method=tf.image.ResizeMethod.BILINEAR)
    img = tf.cast(img, tf.float32)
    if PREPROCESS.lower() == "resnet50":
        img = resnet50.preprocess_input(img)
    elif PREPROCESS.lower() == "efficientnet":
        img = efficientnet.preprocess_input(img)
    else:
        img = img / 255.0
    return img

# Update preprocess_for_single to return a dictionary matching the model's input structure
# def preprocess_for_single(path: str) -> dict:
#     img = tf.io.read_file(path)
#     img = tf.image.decode_image(img, channels=3, expand_animations=False)
#     img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE), method=tf.image.ResizeMethod.BILINEAR)
#     img = tf.cast(img, tf.float32)
#     if PREPROCESS.lower() == "resnet50":
#         from tensorflow.keras.applications.resnet50 import preprocess_input
#         img = preprocess_input(img)
#     elif PREPROCESS.lower() == "efficientnet":
#         from tensorflow.keras.applications.efficientnet import preprocess_input
#         img = efficientnet.preprocess_input(img)
#     else:
#         img = img / 255.0
#     # Assuming the model's input layer is named 'image'
#     # You might need to inspect model.input_names to get the correct name
#     input_name = model.input_names[0] if model.input_names else None
#     if input_name:
#         return {input_name: tf.expand_dims(img, 0)}
#     else:
#         # Fallback for models without named inputs (less common)
#         return tf.expand_dims(img, 0)
# def preprocess_for_single(path: str) -> tf.Tensor:
#     img = tf.io.read_file(path)
#     img = tf.image.decode_image(img, channels=3, expand_animations=False)  # dynamic shape
#     img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE), method=tf.image.ResizeMethod.BILINEAR)

#     # Use the model's dtype:
#     need_dtype = model.inputs[0].dtype  # tf.float32 or tf.uint8

#     if need_dtype == tf.uint8:
#         # Many models with built-in KPLs expect raw uint8 in [0,255]
#         img = tf.clip_by_value(img, 0, 255)
#         img = tf.cast(img, tf.uint8)
#     else:
#         img = tf.cast(img, tf.float32)
#         # Only apply app-specific preprocessing if the model expects float32 tensors
#         if PREPROCESS.lower() == "resnet50":
#             from keras.applications.resnet50 import preprocess_input
#             img = preprocess_input(img)
#         elif PREPROCESS.lower() == "efficientnet":
#             from keras.applications.efficientnet import preprocess_input
#             img = preprocess_input(img)
#         else:
#             img = img / 255.0

#     # Set static shapes (important for some Keras3 graphs)
#     img.set_shape([IMG_SIZE, IMG_SIZE, 3])
#     x = tf.expand_dims(img, 0)
#     x.set_shape([1, IMG_SIZE, IMG_SIZE, 3])
#     return x
# def preprocess_for_single(path: str) -> tf.Tensor:
#     img = tf.io.read_file(path)
#     img = tf.image.decode_image(img, channels=3, expand_animations=False)
#     img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE), method=tf.image.ResizeMethod.BILINEAR)

#     need_dtype = model.inputs[0].dtype  # tf.float32 or tf.uint8

#     if need_dtype == tf.uint8:
#         img = tf.clip_by_value(img, 0, 255)
#         img = tf.cast(img, tf.uint8)
#     else:
#         img = tf.cast(img, tf.float32)
#         # Use keras.applications (not tensorflow.keras) in Keras 3
#         if PREPROCESS.lower() == "resnet50":
#             from keras.applications.resnet50 import preprocess_input
#             img = preprocess_input(img)
#         elif PREPROCESS.lower() == "efficientnet":
#             from keras.applications.efficientnet import preprocess_input
#             img = preprocess_input(img)
#         else:
#             img = img / 255.0

#     # Pin static shape (helps some Keras 3 graphs)
#     img.set_shape([IMG_SIZE, IMG_SIZE, 3])
#     x = tf.expand_dims(img, 0)
#     x.set_shape([1, IMG_SIZE, IMG_SIZE, 3])
#     return x
# def preprocess_for_single(path: str) -> tf.Tensor:
#     img = tf.io.read_file(path)
#     img = tf.image.decode_image(img, channels=3, expand_animations=False)
#     img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE), method=tf.image.ResizeMethod.BILINEAR)

#     need_dtype = model.inputs[0].dtype  # prints showed float32 for you

#     if need_dtype == tf.uint8:
#         img = tf.clip_by_value(img, 0, 255)
#         img = tf.cast(img, tf.uint8)
#     else:
#         img = tf.cast(img, tf.float32)
#         # Use keras.* on Keras 3
#         if PREPROCESS.lower() == "resnet50":
#             from keras.applications.resnet50 import preprocess_input
#             img = preprocess_input(img)
#         elif PREPROCESS.lower() == "efficientnet":
#             from keras.applications.efficientnet import preprocess_input
#             img = preprocess_input(img)
#         else:
#             img = img / 255.0

#     img.set_shape([IMG_SIZE, IMG_SIZE, 3])
#     x = tf.expand_dims(img, 0)
#     x.set_shape([1, IMG_SIZE, IMG_SIZE, 3])
#     return x
from keras.models import load_model
import json
from pathlib import Path
import tensorflow as tf
from keras import Model

# MODEL_PATH = Path("models/resnet50_profilepic_classifier.keras")
model = load_model(MODEL_PATH, compile=False)

# Read metadata
meta = json.loads(MODEL_PATH.with_suffix(".meta.json").read_text())
CLASS_NAMES = json.loads(MODEL_PATH.with_suffix(".classes.json").read_text())
IMG_SIZE = meta["img_size"]
LAST_CONV = meta.get("last_conv", "conv5_block3_out")

def preprocess_for_single(path: str) -> tf.Tensor:
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE), method=tf.image.ResizeMethod.BILINEAR)
    img = tf.cast(img, tf.float32)          # model handles preprocess inside
    img.set_shape([IMG_SIZE, IMG_SIZE, 3])
    x = tf.expand_dims(img, 0)
    x.set_shape([1, IMG_SIZE, IMG_SIZE, 3])
    return x





def build_ds(paths: List[str], labels: List[int], batch=BATCH, shuffle=False) -> tf.data.Dataset:
    x = tf.constant(paths, dtype=tf.string) # Explicitly cast paths to tf.string
    y = tf.constant(labels, dtype=tf.int32)
    ds = tf.data.Dataset.from_tensor_slices((x,y))
    if shuffle:
        ds = ds.shuffle(len(paths), reshuffle_each_iteration=False)
    ds = ds.map(lambda p,l: (preprocess_image(p), tf.one_hot(l, depth=len(CLASS_NAMES))),
                num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch).prefetch(tf.data.AUTOTUNE)
    return ds

paths = df["path"].tolist()
labels = [label_to_index[l] for l in df["label"].tolist()]
ds_eval = build_ds(paths, labels, batch=BATCH, shuffle=False)


## 6) Predict & Build a Score Table


In [26]:
probs = []
for xb, yb in ds_eval:
    p = model.predict(xb, verbose=0)
    probs.append(p)
probs = np.vstack(probs)

pred_idx = probs.argmax(axis=1)
pred_lbl = [index_to_label[i] for i in pred_idx]
true_lbl = df["label"].tolist()

conf = probs[np.arange(len(probs)), pred_idx]

score_df = pd.DataFrame({
    "path": paths,
    "true_label": true_lbl,
    "pred_label": pred_lbl,
    "pred_idx": pred_idx,
    "true_idx": [label_to_index[t] for t in true_lbl],
    "confidence": conf
})
score_df["is_correct"] = score_df["true_label"] == score_df["pred_label"]

score_df.head()


,path,true_label,pred_label,pred_idx,true_idx,confidence,is_correct
0,final/val/animal/df7b903a20dcb96f.jpg,animal,human,2,0,1.0,False
1,final/val/animal/d32586fd66f8e342.jpg,animal,human,2,0,1.0,False
2,final/val/animal/844ac04c499e79c3.jpg,animal,human,2,0,1.0,False
3,final/val/animal/9744f159355e54f2.jpg,animal,human,2,0,1.0,False
4,final/val/animal/f50b6f9bbbf4a9c0.jpg,animal,human,2,0,1.0,False


In [27]:
print("Checking ds_eval contents:")
count = 0
for images, labels in ds_eval.take(5): # Take up to 5 batches to inspect
    print(f"  Batch {count}:")
    print(f"    Images shape: {images.shape}, dtype: {images.dtype}")
    print(f"    Labels shape: {labels.shape}, dtype: {labels.dtype}")
    count += 1

if count == 0:
    print("  ds_eval is empty. No batches were yielded.")
else:
    print(f"  Iterated through {count} batches.")

Checking ds_eval contents:
  Batch 0:
    Images shape: (32, 224, 224, 3), dtype: <dtype: 'float32'>
    Labels shape: (32, 3), dtype: <dtype: 'float32'>
  Batch 1:
    Images shape: (32, 224, 224, 3), dtype: <dtype: 'float32'>
    Labels shape: (32, 3), dtype: <dtype: 'float32'>
  Batch 2:
    Images shape: (32, 224, 224, 3), dtype: <dtype: 'float32'>
    Labels shape: (32, 3), dtype: <dtype: 'float32'>
  Batch 3:
    Images shape: (32, 224, 224, 3), dtype: <dtype: 'float32'>
    Labels shape: (32, 3), dtype: <dtype: 'float32'>
  Batch 4:
    Images shape: (32, 224, 224, 3), dtype: <dtype: 'float32'>
    Labels shape: (32, 3), dtype: <dtype: 'float32'>
  Iterated through 5 batches.


In [28]:
# === lock input info ===
INPUT_KEY   = model.inputs[0].name.split(":")[0]   # e.g., "image"
INPUT_DTYPE = model.inputs[0].dtype                # e.g., tf.float32
print("INPUT_KEY =", INPUT_KEY, "| dtype =", INPUT_DTYPE)

# === preprocessing (keras.* imports; pin shapes) ===
def preprocess_for_single(path: str) -> tf.Tensor:
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE), method=tf.image.ResizeMethod.BILINEAR)

    if INPUT_DTYPE == tf.uint8:
        img = tf.clip_by_value(img, 0, 255)
        img = tf.cast(img, tf.uint8)
    else:
        img = tf.cast(img, tf.float32)
        if PREPROCESS.lower() == "resnet50":
            from keras.applications.resnet50 import preprocess_input
            img = preprocess_input(img)
        elif PREPROCESS.lower() == "efficientnet":
            from keras.applications.efficientnet import preprocess_input
            img = preprocess_input(img)
        else:
            img = img / 255.0

    img.set_shape([IMG_SIZE, IMG_SIZE, 3])
    x = tf.expand_dims(img, 0)
    x.set_shape([1, IMG_SIZE, IMG_SIZE, 3])
    return x

# === use the same successful call style for ANY model (base or grad_model) ===
def call_model_anyway(m, x):
    try:
        return m(x, training=False)  # PLAIN TENSOR (this works for your base model)
    except Exception:
        pass
    try:
        name = m.inputs[0].name.split(":")[0]
        return m({name: x}, training=False)
    except Exception:
        pass
    try:
        return m([x], training=False)
    except Exception:
        pass
    raise RuntimeError("Unable to call the model with tensor/dict/list.")

# === pick a 4D feature map layer (resnet50 default first, then fallback) ===
from keras import Model

def get_last_conv_layer_name(m) -> str:
    for name in ("conv5_block3_out", "post_relu", "conv_7b", "top_activation"):
        try:
            out = m.get_layer(name).output
            if hasattr(out, "shape") and len(out.shape) == 4:
                return name
        except Exception:
            pass
    for layer in reversed(m.layers):
        try:
            out = getattr(layer, "output", None)
            if out is not None and hasattr(out, "shape") and len(out.shape) == 4:
                return layer.name
        except Exception:
            pass
    raise ValueError("No 4D conv/feature layer found for Grad-CAM.")

def make_gradcam_heatmap(x: tf.Tensor,
                         class_index: int | None = None,
                         last_conv_layer_name: str = LAST_CONV):
    conv_layer = model.get_layer(last_conv_layer_name)
    grad_model = Model(inputs=model.inputs, outputs=[conv_layer.output, model.output])
    with tf.GradientTape() as tape:
        tape.watch(x)
        conv_outputs, preds = grad_model(x, training=False)   # PLAIN TENSOR CALL
        if class_index is None:
            class_index = int(tf.argmax(preds[0]))
        target = preds[:, class_index]
    grads = tape.gradient(target, conv_outputs)
    pooled = tf.reduce_mean(grads, axis=(0,1,2))
    fmap   = conv_outputs[0]
    cam    = tf.reduce_sum(fmap * pooled, axis=-1)
    cam    = tf.maximum(cam, 0.0)
    cam   /= tf.reduce_max(cam) if tf.reduce_max(cam) > 0 else 1.0
    return cam.numpy()

# # === Grad-CAM that calls the submodel with the same (tensor) style ===
# def make_gradcam_heatmap(
#     x: tf.Tensor,
#     class_index: int | None = None,
#     last_conv_layer_name: str | None = None,
# ):
#     if last_conv_layer_name is None:
#         last_conv_layer_name = get_last_conv_layer_name(model)

#     conv_layer = model.get_layer(last_conv_layer_name)

#     # Build submodel; DO NOT call it with a dict — use call_model_anyway(...)
#     grad_model = Model(inputs=model.inputs, outputs=[conv_layer.output, model.output])

#     with tf.GradientTape() as tape:
#         tape.watch(x)
#         conv_outputs, preds = call_model_anyway(grad_model, x)  # <-- plain tensor first
#         if class_index is None:
#             class_index = int(tf.argmax(preds[0]))
#         target = preds[:, class_index]

#     grads = tf.gradients(target, conv_outputs)[0] if hasattr(tf, "gradients") else tf.GradientTape().gradient(target, conv_outputs)
#     if grads is None:  # fallback using active tape context
#         grads = tf.GradientTape().gradient(target, conv_outputs)

#     pooled = tf.reduce_mean(grads, axis=(0, 1, 2))      # (C,)
#     fmap  = conv_outputs[0]                              # (H, W, C)
#     cam   = tf.reduce_sum(fmap * pooled, axis=-1)       # (H, W)

#     cam = tf.maximum(cam, 0.0)
#     mx  = tf.reduce_max(cam)
#     cam = tf.where(mx > 0, cam / mx, tf.zeros_like(cam))
#     return cam.numpy()

# === sanity probes (note: DO NOT use dict calls here) ===
xp = preprocess_for_single(score_df.iloc[0].path)
_  = call_model_anyway(model, xp)          # base model forward
print("Base forward OK")
_  = make_gradcam_heatmap(xp)              # grad-cam forward
print("Grad-CAM probe OK")


INPUT_KEY = image | dtype = float32
Base forward OK


ValueError: No such layer: conv5_block3_out. Existing layers are: ['image', 'aug', 'resnet50', 'global_average_pooling2d_9', 'dropout_18', 'dense_18', 'dropout_19', 'dense_19'].


## 7) Grad‑CAM Utilities


In [ ]:
def find_last_conv_layer(model: tf.keras.Model) -> str:
    for layer in reversed(model.layers):
        try:
            out_shape = layer.output_shape
            if isinstance(out_shape, list):
                out_shape = out_shape[0]
            if len(out_shape) == 4:
                return layer.name
        except Exception:
            continue
    raise ValueError("No 4D conv layer found; set TARGET_LAYER_NAME manually.")

last_conv_name = TARGET_LAYER_NAME or find_last_conv_layer(model)
last_conv = model.get_layer(last_conv_name)
print("Grad‑CAM target layer:", last_conv_name)

# Build grad_model exactly from the original model's inputs/outputs
# This handles potential issues with TFSMLayer inputs/outputs
grad_model = tf.keras.models.Model(inputs=model.inputs, outputs=[last_conv.output, model.output])

def _first_input_name(m):
    try:
        # Keras 3: m.inputs is a list of KerasTensors with .name like 'image:0'
        return m.inputs[0].name.split(":")[0]
    except Exception:
        names = getattr(m, "input_names", None)
        if names and len(names) > 0:
            return names[0]
    return None

def _call_with_structured_inputs(m, x):
    # 1) Plain tensor call
    try:
        return m(x, training=False)
    except Exception:
        print('plain tensor call error')
        # pass
    # 2) Dict by input name
    name = _first_input_name(m)
    if name:
        try:
            return m({name: x}, training=False)
        except Exception:
            print('dict by input name error')
            # pass
    # 3) Single-element list
    try:
        return m([x], training=False)
    except Exception:
        print('single-element list error')
        # pass
    # 4) Clear error
    raise RuntimeError(
        f"Could not call model with structured inputs. "
        f"Input name guessed={name}; got tensor shape={getattr(x, 'shape', None)}"
    )

# def _call_with_structured_inputs(m, x):
#     """
#     Try calling model with bare tensor, [tensor], and {input_name: tensor}.
#     Works around Keras 3 structured-input expectations (e.g., Input(name="image", ...)).
#     """
#     # 1) Try bare tensor
#     try:
#         return m(x, training=False)
#     except Exception:
#         pass
#     # 2) Try list-wrapped (single-input models often accept [x])
#     try:
#         return m([x], training=False)
#     except Exception:
#         pass
#     # 3) Try dict by first input name
#     try:
#         names = getattr(m, "input_names", None)
#         if names and len(names) == 1:
#             return m({names[0]: x}, training=False)
#     except Exception:
#         pass
#     # If still failing, raise a clear error
#     raise RuntimeError(
#         f"Could not call model with structured inputs. "
#         f"Input names={getattr(m, 'input_names', None)}; got tensor shape={x.shape}"
#     )


# def make_gradcam_heatmap(img_tensor: tf.Tensor, class_index: int) -> np.ndarray:
    # with tf.GradientTape() as tape:
    #     # Use the helper function to call the grad_model robustly
    #     conv_out, preds = _call_with_structured_inputs(grad_model, img_tensor)

    #     # The model.output might be a list or a single tensor depending on how it was saved/loaded.
    #     # We need the actual prediction tensor for calculating the target gradient.
    #     # Access the prediction tensor - assuming it's the second output of grad_model
    #     # If preds is a tuple/list from the model outputs, get the actual prediction tensor
    #     model_output = preds[0] if isinstance(preds, (list, tuple)) else preds


    #     if class_index is None:
    #         class_index = tf.argmax(model_output[0])

    #     # Calculate the target score using one-hot encoding and reduce_sum
    #     target_class_one_hot = tf.one_hot([class_index], depth=tf.shape(model_output)[1])
    #     target = tf.reduce_sum(model_output * target_class_one_hot, axis=1)


    # grads = tape.gradient(target, conv_out)
    # pooled_grads = tf.reduce_mean(grads, axis=(1,2))
    # conv_out = conv_out[0]
    # pooled_grads = pooled_grads[0]

    # heatmap = tf.tensordot(conv_out, pooled_grads, axes=(2,0))
    # heatmap = tf.maximum(heatmap, 0)
    # heatmap = heatmap / (tf.reduce_max(heatmap) + 1e-8)
    # return heatmap.numpy()
# def make_gradcam_heatmap(x: tf.Tensor,
#                          class_index: int | None = None,
#                          model: tf.keras.Model = model,
#                          last_conv_layer_name: str = LAST_CONV_LAYER_NAME):
#     # Build a grad model that returns conv outputs + predictions
#     base_in = model.inputs  # preserves named inputs
#     conv_out = model.get_layer(last_conv_layer_name).output
#     grad_model = tf.keras.Model(inputs=base_in, outputs=[conv_out, model.output])

#     with tf.GradientTape() as tape:
#         # Ensure tape sees the input
#         tape.watch(x)
#         conv_outputs, preds = call_model_anyway(grad_model, x)
#         if class_index is None:
#             class_index = int(tf.argmax(preds[0]))
#         target = preds[:, class_index]

#     grads = tape.gradient(target, conv_outputs)               # (1, H, W, C)
#     pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))      # (C,)
#     conv_outputs = conv_outputs[0]                            # (H, W, C)
#     heatmap = tf.reduce_sum(conv_outputs * pooled_grads, axis=-1)

#     # Normalize to [0,1]
#     heatmap = tf.maximum(heatmap, 0.0)
#     denom = tf.reduce_max(heatmap)
#     heatmap = tf.where(denom > 0, heatmap / denom, tf.zeros_like(heatmap))
#     return heatmap.numpy()
# --- 1) Helper to list & pick a good target conv layer ----------------------
# def get_last_conv_layer_name(m) -> str:
#     # Try common names first (ResNet/EfficientNet variants)
#     for name in ("conv5_block3_out", "post_relu", "conv_7b", "top_activation"):
#         try:
#             layer = m.get_layer(name)
#             out = getattr(layer, "output", None)
#             if out is not None and hasattr(out, "shape") and len(out.shape) == 4:
#                 return name
#         except Exception:
#             pass

#     # Generic fallback: scan layers from the end for any 4D feature map
#     for layer in reversed(m.layers):
#         try:
#             out = getattr(layer, "output", None)
#             if out is not None and hasattr(out, "shape") and len(out.shape) == 4:
#                 return layer.name
#         except Exception:
#             continue

#     raise ValueError("Couldn't find a 4D conv/feature layer to use for Grad-CAM.")

# def list_4d_feature_layers(m, limit=25):
#     names = []
#     for layer in m.layers:
#         try:
#             out = getattr(layer, "output", None)
#             if out is not None and hasattr(out, "shape") and len(out.shape) == 4:
#                 names.append(layer.name)
#         except Exception:
#             pass
#     print(names[-limit:])
#     return names

# # Optional: see what's available
# # _ = list_4d_feature_layers(model)

# # If you want a constant, you can set it explicitly (ResNet50 default):
# # LAST_CONV_LAYER_NAME = "conv5_block3_out"


# # --- 2) Grad-CAM that auto-discovers the target layer if not provided -------
# def make_gradcam_heatmap(
#     x: tf.Tensor,
#     class_index: int | None = None,
#     model: tf.keras.Model = model,
#     last_conv_layer_name: str | None = None,
# ):
#     if last_conv_layer_name is None:
#         last_conv_layer_name = get_last_conv_layer_name(model)

#     conv_layer = model.get_layer(last_conv_layer_name)

#     # Build a grad model that preserves the model's real inputs
#     grad_model = tf.keras.Model(inputs=model.inputs,
#                                 outputs=[conv_layer.output, model.output])

#     with tf.GradientTape() as tape:
#         tape.watch(x)
#         conv_outputs, preds = call_model_anyway(grad_model, x)
#         if class_index is None:
#             class_index = int(tf.argmax(preds[0]))
#         target = preds[:, class_index]

#     grads = tape.gradient(target, conv_outputs)               # (1, H, W, C)
#     pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))      # (C,)
#     conv_map = conv_outputs[0]                                # (H, W, C)
#     heatmap = tf.reduce_sum(conv_map * pooled_grads, axis=-1) # (H, W)

#     heatmap = tf.maximum(heatmap, 0.0)
#     denom = tf.reduce_max(heatmap)
#     heatmap = tf.where(denom > 0, heatmap / denom, tf.zeros_like(heatmap))
#     return heatmap.numpy()


# def overlay_heatmap_on_image(path: str, heatmap: np.ndarray, alpha=ALPHA, img_size=IMG_SIZE):
#     img = Image.open(path).convert("RGB").resize((img_size, img_size))
#     img_np = np.array(img)
#     hm = cv2.resize(heatmap, (img_size, img_size))
#     hm = np.uint8(255 * hm)
#     hm_color = cv2.applyColorMap(hm, cv2.COLORMAP_JET)
#     overlay = cv2.addWeighted(hm_color, alpha, img_np[:, :, ::-1], 1.0, 0)
#     overlay = overlay[:, :, ::-1]
#     return img_np, overlay

# def border_attention_fraction(heatmap: np.ndarray, border_ratio: float = 0.08) -> float:
#     h, w = heatmap.shape
#     b = int(round(min(h, w) * border_ratio))
#     core = heatmap[b:h-b, b:w-b].sum() if (b>0 and (h-2*b)>0 and (w-2*b)>0) else 0.0
#     total = heatmap.sum() + 1e-8
#     border = total - core
#     return float(border / total)
from keras import Model  # avoid tensorflow.keras on Keras 3+

def get_last_conv_layer_name(m) -> str:
    # Try common names, else fallback to the last 4D feature map layer
    for name in ("conv5_block3_out", "post_relu", "conv_7b", "top_activation"):
        try:
            out = m.get_layer(name).output
            if hasattr(out, "shape") and len(out.shape) == 4:
                return name
        except Exception:
            pass
    for layer in reversed(m.layers):
        try:
            out = getattr(layer, "output", None)
            if out is not None and hasattr(out, "shape") and len(out.shape) == 4:
                return layer.name
        except Exception:
            pass
    raise ValueError("No 4D conv/feature layer found for Grad-CAM.")

def make_gradcam_heatmap(
    x: tf.Tensor,
    class_index: int | None = None,
    last_conv_layer_name: str | None = None,
):
    if last_conv_layer_name is None:
        last_conv_layer_name = get_last_conv_layer_name(model)

    conv_layer = model.get_layer(last_conv_layer_name)

    # Build a grad model that preserves the original input structure
    grad_model = Model(inputs=model.inputs, outputs=[conv_layer.output, model.output])

    with tf.GradientTape() as tape:
        tape.watch(x)  # make sure gradients can flow wrt the input image
        # *** KEY CHANGE: call with explicit dict keyed by INPUT_KEY ***
        conv_outputs, preds = grad_model({INPUT_KEY: x}, training=False)

        if class_index is None:
            class_index = int(tf.argmax(preds[0]))
        target = preds[:, class_index]

    grads = tape.gradient(target, conv_outputs)               # (1, H, W, C)
    pooled = tf.reduce_mean(grads, axis=(0, 1, 2))            # (C,)
    fmap  = conv_outputs[0]                                   # (H, W, C)
    cam   = tf.reduce_sum(fmap * pooled, axis=-1)             # (H, W)

    cam = tf.maximum(cam, 0.0)
    mx  = tf.reduce_max(cam)
    cam = tf.where(mx > 0, cam / mx, tf.zeros_like(cam))
    return cam.numpy()


In [ ]:
def call_model_anyway(m, x):
    # 1) Plain call
    try:
        return m(x, training=False)
    except Exception:
        pass
    # 2) Dict by input name
    try:
        name = m.inputs[0].name.split(":")[0]
        return m({name: x}, training=False)
    except Exception:
        pass
    # 3) Single-element list
    try:
        return m([x], training=False)
    except Exception:
        pass
    raise RuntimeError("Unable to call the model with tensor/dict/list.")

test_path = score_df.iloc[0].path
x = preprocess_for_single(test_path)
_ = call_model_anyway(model, x)  # should not raise
print("Single forward pass OK")

print("Model inputs:", [t.name for t in model.inputs], [t.dtype for t in model.inputs])
xp = preprocess_for_single(score_df.iloc[0].path)

# Base model sanity (should pass):
_ = model({INPUT_KEY: xp}, training=False)

# Grad-CAM sanity (should pass):
_ = make_gradcam_heatmap(xp)
print("Grad-CAM probe OK")



## 8) Visualize Grad‑CAM Panels


In [ ]:
import math

def pick_samples(score_df: pd.DataFrame, per_class_mis:int, per_class_ok:int):
    rows = []
    for cls in CLASS_NAMES:
        sub = score_df[score_df["true_label"] == cls].copy()
        mis = sub[~sub["is_correct"]].sort_values("confidence", ascending=False).head(per_class_mis)
        ok  = sub[sub["is_correct"]].sort_values("confidence", ascending=False).head(per_class_ok)
        rows.append(("MIS", cls, mis))
        rows.append(("OK",  cls, ok))
    return rows

# def preprocess_for_single(path: str) -> tf.Tensor:
#     img = tf.io.read_file(path)
#     img = tf.image.decode_image(img, channels=3, expand_animations=False)
#     img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE), method=tf.image.ResizeMethod.BILINEAR)
#     img = tf.cast(img, tf.float32)
#     if PREPROCESS.lower() == "resnet50":
#         from tensorflow.keras.applications.resnet50 import preprocess_input
#         img = preprocess_input(img)
#     elif PREPROCESS.lower() == "efficientnet":
#         from tensorflow.keras.applications.efficientnet import preprocess_input
#         img = preprocess_input(img)
#     else:
#         img = img / 255.0
#     return tf.expand_dims(img, 0)

# 0) After loading the model
INPUT_KEY   = model.inputs[0].name.split(":")[0]   # e.g., "image"
INPUT_DTYPE = model.inputs[0].dtype                # e.g., tf.float32
print("INPUT_KEY =", INPUT_KEY, "| dtype =", INPUT_DTYPE)

# 1) Preprocess to match dtype + static shapes (Keras 3 friendly)
def preprocess_for_single(path: str) -> tf.Tensor:
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE), method=tf.image.ResizeMethod.BILINEAR)

    if INPUT_DTYPE == tf.uint8:
        img = tf.clip_by_value(img, 0, 255)
        img = tf.cast(img, tf.uint8)
    else:
        img = tf.cast(img, tf.float32)
        # Use keras.* with Keras 3
        if PREPROCESS.lower() == "resnet50":
            from keras.applications.resnet50 import preprocess_input
            img = preprocess_input(img)
        elif PREPROCESS.lower() == "efficientnet":
            from keras.applications.efficientnet import preprocess_input
            img = preprocess_input(img)
        else:
            img = img / 255.0

    img.set_shape([IMG_SIZE, IMG_SIZE, 3])
    x = tf.expand_dims(img, 0)
    x.set_shape([1, IMG_SIZE, IMG_SIZE, 3])
    return x

# 2) Detect the working call style ONCE using a sample
def detect_call_style(m, sample_x):
    try:
        _ = m(sample_x, training=False)
        return "tensor"
    except Exception:
        pass
    try:
        _ = m({INPUT_KEY: sample_x}, training=False)
        return "dict"
    except Exception:
        pass
    try:
        _ = m([sample_x], training=False)
        return "list"
    except Exception:
        pass
    raise RuntimeError("No working call style found for the model.")

# Build a sample input to probe
_probe_x = preprocess_for_single(score_df.iloc[0].path)
CALL_STYLE = detect_call_style(model, _probe_x)
print("CALL_STYLE =", CALL_STYLE)

def call_like(m, x):
    if CALL_STYLE == "tensor":
        return m(x, training=False)
    elif CALL_STYLE == "dict":
        return m({INPUT_KEY: x}, training=False)
    else:  # "list"
        return m([x], training=False)

# 3) Grad-CAM utils (auto-pick a 4D feature map layer)
from keras import Model

def get_last_conv_layer_name(m) -> str:
    for name in ("conv5_block3_out", "post_relu", "conv_7b", "top_activation"):
        try:
            out = m.get_layer(name).output
            if hasattr(out, "shape") and len(out.shape) == 4:
                return name
        except Exception:
            pass
    for layer in reversed(m.layers):
        try:
            out = getattr(layer, "output", None)
            if out is not None and hasattr(out, "shape") and len(out.shape) == 4:
                return layer.name
        except Exception:
            pass
    raise ValueError("No 4D conv/feature layer found for Grad-CAM.")

def make_gradcam_heatmap(
    x: tf.Tensor,
    class_index: int | None = None,
    last_conv_layer_name: str | None = None,
):
    if last_conv_layer_name is None:
        last_conv_layer_name = get_last_conv_layer_name(model)

    conv_layer = model.get_layer(last_conv_layer_name)

    # Build grad model that INHERITS the same input structure as model
    grad_model = Model(inputs=model.inputs, outputs=[conv_layer.output, model.output])

    with tf.GradientTape() as tape:
        tape.watch(x)
        # <<< KEY CHANGE: call the grad_model using the same CALL_STYLE as base >>>
        conv_outputs, preds = call_like(grad_model, x)
        if class_index is None:
            class_index = int(tf.argmax(preds[0]))
        target = preds[:, class_index]

    grads = tape.gradient(target, conv_outputs)               # (1, H, W, C)
    pooled = tf.reduce_mean(grads, axis=(0, 1, 2))            # (C,)
    fmap  = conv_outputs[0]                                   # (H, W, C)
    cam   = tf.reduce_sum(fmap * pooled, axis=-1)             # (H, W)

    cam = tf.maximum(cam, 0.0)
    mx  = tf.reduce_max(cam)
    cam = tf.where(mx > 0, cam / mx, tf.zeros_like(cam))
    return cam.numpy()

# 4) Sanity probes
_ = call_like(model, _probe_x)                 # base model forward
_ = make_gradcam_heatmap(_probe_x)             # grad-cam forward
print("Grad-CAM probe OK")


def panel_for_group(kind: str, cls: str, group_df: pd.DataFrame, save_path: Path):
    n = len(group_df)
    if n == 0:
        return

    cols = 3
    rows = n
    fig_h = max(4, rows * 3)
    fig_w = 12

    plt.figure(figsize=(fig_w, fig_h))
    idx = 1
    records = []

    # TEST TEST TEST
    print("Model inputs:", [t.name for t in model.inputs], [t.dtype for t in model.inputs])
    xp = preprocess_for_single(score_df.iloc[0].path)
    _ = call_model_anyway(model, xp)   # must pass
    print("Grad-CAM probe...")
    _ = make_gradcam_heatmap(xp, class_index=None)   # should pass, too


    for r in group_df.itertuples(index=False):
        # x = preprocess_for_single(r.path)
        # heat = make_gradcam_heatmap(x, class_index=r.pred_idx)
        # img_np, overlay = overlay_heatmap_on_image(r.path, heat)
        for r in group_df.itertuples(index=False):
          x = preprocess_for_single(r.path)
          # If you predict elsewhere, always:
          # preds = call_model_anyway(model, x)
          heat = make_gradcam_heatmap(x, class_index=r.pred_idx)
        frac = border_attention_fraction(heat)

        plt.subplot(n, cols, idx);   plt.imshow(img_np);  plt.axis("off");
        plt.title(f"Orig\ntrue={r.true_label}\npred={r.pred_label}\nconf={r.confidence:.2f}")
        idx += 1
        plt.subplot(n, cols, idx);   plt.imshow(heat, cmap="jet");  plt.axis("off");  plt.title("Heatmap")
        idx += 1
        plt.subplot(n, cols, idx);   plt.imshow(overlay); plt.axis("off");
        plt.title(f"Overlay\nborder={frac:.2f}")
        idx += 1

        records.append({
            "path": r.path,
            "true_label": r.true_label,
            "pred_label": r.pred_label,
            "confidence": r.confidence,
            "border_attention_frac": frac,
            "is_correct": r.is_correct
        })

    plt.suptitle(f"{kind}: {cls} — {n} samples", y=1.02)
    plt.tight_layout()
    plt.savefig(save_path, dpi=160, bbox_inches="tight")
    plt.show()

    pd.DataFrame(records).to_csv(save_path.with_suffix(".csv"), index=False)

print("Model inputs:", [t.name for t in model.inputs])  # e.g., ['image:0']

samples = pick_samples(score_df, N_MISCLASS_PER_CLASS, N_CORRECT_PER_CLASS)
for kind, cls, gdf in samples:
    slug = f"{kind.lower()}_{cls}".replace(" ", "_")
    out_file = OUT_DIR / f"gradcam_{slug}.png"
    panel_for_group(kind, cls, gdf, out_file)

print("Saved panels to:", OUT_DIR.resolve())


In [ ]:
def find_last_conv_layer(model: tf.keras.Model) -> str:
    for layer in reversed(model.layers):
        try:
            out_shape = layer.output_shape
            if isinstance(out_shape, list):
                out_shape = out_shape[0]
            if len(out_shape) == 4:
                return layer.name
        except Exception:
            continue
    raise ValueError("No 4D conv layer found; set TARGET_LAYER_NAME manually.")

# We will build the grad_model within the make_gradcam_heatmap function
# to avoid potential issues with the loaded model structure.

def _call_with_structured_inputs(m, x):
    """
    Try calling model with bare tensor, [tensor], and {input_name: tensor}.
    Works around Keras 3 structured-input expectations (e.g., Input(name="image", ...)).
    """
    # 1) Try bare tensor
    try:
        return m(x, training=False)
    except Exception:
        pass
    # 2) Try list-wrapped (single-input models often accept [x])
    try:
        return m([x], training=False)
    except Exception:
        pass
    # 3) Try dict by first input name
    try:
        names = getattr(m, "input_names", None)
        if names and len(names) == 1:
            return m({names[0]: x}, training=False)
    except Exception:
        pass
    # If still failing, raise a clear error
    raise RuntimeError(
        f"Could not call model with structured inputs. "
        f"Input names={getattr(m, 'input_names', None)}; got tensor shape={x.shape}"
    )


def make_gradcam_heatmap(img_tensor: tf.Tensor, class_index: int) -> np.ndarray:
    # Find the last convolutional layer dynamically if not set
    last_conv_name = TARGET_LAYER_NAME or find_last_conv_layer(model)
    print("Using Grad‑CAM target layer:", last_conv_name)

    # Define a temporary model to get the output of the last conv layer and the final output
    # We create this inside the function to work with the loaded model object directly
    try:
        # Attempt to create a functional model from the original model's layers
        grad_model_temp = tf.keras.models.Model(
            inputs=model.inputs,
            outputs=[model.get_layer(last_conv_name).output, model.output]
        )
    except Exception as e:
        # Fallback if creating the functional model fails (e.g., with TFSMLayer)
        # We need a way to get intermediate and final outputs from the loaded model
        # This might require model-specific handling or using an alternative Grad-CAM implementation
        print(f"Warning: Failed to create functional grad_model_temp: {e}")
        print("Attempting a fallback approach (may not work for all model types).")

        # Fallback: Create a dummy model to get the output of the last conv layer
        # This is a less robust approach and might not work depending on the model architecture
        try:
             intermediate_layer_model = tf.keras.models.Model(
                inputs=model.inputs,
                outputs=model.get_layer(last_conv_name).output
             )
        except Exception as e:
             raise RuntimeError(f"Could not create intermediate layer model: {e}")

        # Need to handle how to get both intermediate and final outputs with one call/tape
        # This part is tricky with arbitrary loaded models.
        # Let's try to get the intermediate output and final output separately for now
        # This might break the gradient tape, but let's see.
        with tf.GradientTape() as tape:
            # Need to watch the input tensor if we are calling parts of the model separately
            # tape.watch(img_tensor) # This might be needed depending on how inputs are handled

            # Get the intermediate output
            conv_out = _call_with_structured_inputs(intermediate_layer_model, img_tensor)

            # Get the final predictions from the original model
            preds = _call_with_structured_inputs(model, img_tensor)

            model_output = preds[0] if isinstance(preds, (list, tuple)) else preds


            if class_index is None:
                class_index = tf.argmax(model_output[0])

            target_class_one_hot = tf.one_hot([class_index], depth=tf.shape(model_output)[1])
            target = tf.reduce_sum(model_output * target_class_one_hot, axis=1)


        # Calculate gradients with respect to the intermediate output
        # This will likely fail if the intermediate_layer_model breaks the graph
        try:
            grads = tape.gradient(target, conv_out)
        except Exception as e:
            raise RuntimeError(f"Gradient calculation failed. This might be due to the loaded model's structure or how intermediate outputs are accessed: {e}")

        pooled_grads = tf.reduce_mean(grads, axis=(1,2))
        conv_out = conv_out[0]
        pooled_grads = pooled_grads[0]

        heatmap = tf.tensordot(conv_out, pooled_grads, axes=(2,0))
        heatmap = tf.maximum(heatmap, 0)
        heatmap = heatmap / (tf.reduce_max(heatmap) + 1e-8)
        return heatmap.numpy()


    # Original logic if functional model creation works
    with tf.GradientTape() as tape:
        conv_out, preds = _call_with_structured_inputs(grad_model_temp, img_tensor)

        model_output = preds[0] if isinstance(preds, (list, tuple)) else preds

        if class_index is None:
            class_index = tf.argmax(model_output[0])

        target_class_one_hot = tf.one_hot([class_index], depth=tf.shape(model_output)[1])
        target = tf.reduce_sum(model_output * target_class_one_hot, axis=1)


    grads = tape.gradient(target, conv_out)
    pooled_grads = tf.reduce_mean(grads, axis=(1,2))
    conv_out = conv_out[0]
    pooled_grads = pooled_grads[0]

    heatmap = tf.tensordot(conv_out, pooled_grads, axes=(2,0))
    heatmap = tf.maximum(heatmap, 0)
    heatmap = heatmap / (tf.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()


def overlay_heatmap_on_image(path: str, heatmap: np.ndarray, alpha=ALPHA, img_size=IMG_SIZE):
    img = Image.open(path).convert("RGB").resize((img_size, img_size))
    img_np = np.array(img)
    hm = cv2.resize(heatmap, (img_size, img_size))
    hm = np.uint8(255 * hm)
    hm_color = cv2.applyColorMap(hm, cv2.COLORMAP_JET)
    overlay = cv2.addWeighted(hm_color, alpha, img_np[:, :, ::-1], 1.0, 0)
    overlay = overlay[:, :, ::-1]
    return img_np, overlay

def border_attention_fraction(heatmap: np.ndarray, border_ratio: float = 0.08) -> float:
    h, w = heatmap.shape
    b = int(round(min(h, w) * border_ratio))
    core = heatmap[b:h-b, b:w-b].sum() if (b>0 and (h-2*b)>0 and (w-2*b)>0) else 0.0
    total = heatmap.sum() + 1e-8
    border = total - core
    return float(border / total)


## 9) Single‑Image Helper


In [ ]:

def gradcam_single_image(path: str, class_idx: int = None):
    x = preprocess_for_single(path)
    if class_idx is None:
        p = model.predict(x, verbose=0)[0]
        class_idx = int(np.argmax(p))
    heat = make_gradcam_heatmap(x, class_index=class_idx)
    img_np, overlay = overlay_heatmap_on_image(path, heat)
    frac = border_attention_fraction(heat)

    plt.figure(figsize=(10,3))
    plt.subplot(1,3,1); plt.imshow(img_np); plt.axis("off"); plt.title("Original")
    plt.subplot(1,3,2); plt.imshow(heat, cmap="jet"); plt.axis("off"); plt.title("Heatmap")
    plt.subplot(1,3,3); plt.imshow(overlay); plt.axis("off"); plt.title(f"Overlay\nborder={frac:.2f}")
    plt.tight_layout()
    plt.show()

# Example:
# gradcam_single_image(df.iloc[0]['path'])



## 10) What to Look For

- **Correct**: heat concentrated on salient object (face/body/animal).
- **Wrong**: heat on **backgrounds, borders, logos, corner watermarks**, or text.
- If border‑attention fractions are systematically high, consider **masking/cropping** or **augmentations** that randomize edges.


# Tools

In [ ]:
def gcs_auth():
  import json
  from google.colab import userdata

  # Get the service account key from Colab Secrets
  service_account_info = json.loads(userdata.get('GCP_SERVICE_ACCOUNT_KEY'))

  # Define the path to save the service account key file
  key_file_path = 'service_account_key.json'

  # Save the service account key to a file
  with open(key_file_path, 'w') as f:
      json.dump(service_account_info, f)

  # Authenticate gcloud and gsutil using the service account key file
  !gcloud auth activate-service-account --key-file {key_file_path}

In [ ]:
def load_zip_data_from_gcs():
  import time

  start_time = time.time()

  bucket_name = 'gs://along-capstone-data'
  source_directory = 'final_zip' # This is the directory in the bucket
  destination_directory = '.' # This is the local destination directory
  zip_file_name = 'final.zip'

  gcs_auth()

  # Copy the directory containing the zip file
  !gsutil -m -q cp -r {bucket_name}/{source_directory} {destination_directory}

  # Construct the local path to the zip file
  local_zip_file_path = f"{destination_directory}/{source_directory}/{zip_file_name}"

  # Unzip the data directory.
  print("Unzipping data...")
  !unzip -o -q {local_zip_file_path} -d {destination_directory}

  end_time = time.time()
  elapsed_time_seconds = end_time - start_time
  elapsed_time_minutes = int(elapsed_time_seconds // 60)
  elapsed_time_remaining_seconds = int(elapsed_time_seconds % 60)


  print(f"Zip download and unzip elapsed time: {elapsed_time_minutes} minutes and {elapsed_time_remaining_seconds} seconds")

# load_zip_data_from_gcs()

In [ ]:
def load_model_from_gcs():
  import time

  start_time = time.time()

  bucket_name = 'gs://along-capstone-data'
  source_directory = 'models' # This is the directory in the bucket
  destination_directory = '.' # This is the local destination directory

  gcs_auth()

  # Copy the directory containing the zip file
  !gsutil -m -q cp -r {bucket_name}/{source_directory} {destination_directory}

  end_time = time.time()
  elapsed_time_seconds = end_time - start_time
  elapsed_time_minutes = int(elapsed_time_seconds // 60)
  elapsed_time_remaining_seconds = int(elapsed_time_seconds % 60)

  print(f"Model download elapsed time: {elapsed_time_minutes} minutes and {elapsed_time_remaining_seconds} seconds")

# load_model_from_gcs()

In [ ]:
def load_model_from_gcs(bucket_name='gs://along-capstone-data', model_name="resnet50_profilepic_classifier.keras", directory='models'):
  """Saves a Keras model to a Google Cloud Storage bucket."""
  import json
  from google.colab import userdata
  from pathlib import Path
  import os

  # Get the service account key from Colab Secrets
  service_account_info = json.loads(userdata.get('GCP_SERVICE_ACCOUNT_KEY'))

  # Define the path to save the service account key file
  key_file_path = 'service_account_key.json'

  # Save the service account key to a file
  with open(key_file_path, 'w') as f:
      json.dump(service_account_info, f)

  # Authenticate gcloud and gsutil using the service account key file
  !gcloud auth activate-service-account --key-file {key_file_path}

  # Define the local path to save the model temporarily
  local_model_dir = Path(directory)
  local_model_dir.mkdir(parents=True, exist_ok=True)
  # local_model_path = local_model_dir / model_name

  # Load the model locally
  # model_to_save.save(local_model_path)
  # print(f"Model saved locally to {local_model_path}")

  # Upload the model to GCS
  gcs_model_path = f"{bucket_name}/{directory}/{model_name}"
  !gsutil cp {gcs_model_path} {local_model_dir}
  print(f"Model loaded to {gcs_model_path}")

  # # Clean up the local model file and directory
  # local_model_path.unlink()
  # local_model_dir.rmdir() # This will only work if the directory is empty after deleting the model file
  # print(f"Local model file {local_model_path} and directory {local_model_dir} removed.")

# Example call (uncomment to use):
# load_model_from_gcs()

# Probably for Delete

In [ ]:
# Install the Google Cloud Storage client library
# !pip install google-cloud-storage

Authenticating with the service account key and downloading files from the bucket.

Using `gsutil` with multiple threads for faster downloads.

## Load Data (Pictures) from GCS

In [ ]:
# def load_files():
#   # Specify the bucket name and the local directory to save the files
#   bucket_name = 'gs://along-capstone-data'
#   source_directory = 'final'
#   destination_directory = 'data'

#   # Create the destination directory if it doesn't exist
#   import os
#   os.makedirs(destination_directory, exist_ok=True)

#   import json
#   from google.colab import userdata
#   import time

#   # Get the service account key from Colab Secrets
#   service_account_info = json.loads(userdata.get('GCP_SERVICE_ACCOUNT_KEY'))

#   # Define the path to save the service account key file
#   key_file_path = 'service_account_key.json'

#   # Save the service account key to a file
#   with open(key_file_path, 'w') as f:
#       json.dump(service_account_info, f)

#   # Authenticate gcloud and gsutil using the service account key file
#   !gcloud auth activate-service-account --key-file {key_file_path}

#   start_time = time.time()

#   # Download files using gsutil with multiple threads
#   # -m enables multithreading
#   # -r recursively copies directories and files
#   !gsutil -m cp -r {bucket_name}/{source_directory} {destination_directory}

#   end_time = time.time()
#   elapsed_time = end_time - start_time

#   print("Download complete.")
#   print(f"Load operation took {elapsed_time:.2f} seconds.")


# load_files()

## Load Model from GCS

In [ ]:
# # === GradcamInspection: DROP-IN REPLACEMENT for model + metadata loading ===
# from pathlib import Path
# from keras.models import load_model
# import tensorflow as tf
# import json

# def load_model_and_meta(model_path: str | Path):
#     p = Path(model_path)
#     if not p.exists():
#         raise FileNotFoundError(f"Model file not found: {p.resolve()}")

#     # Load the Keras single-file model (no compile needed for inference)
#     model = load_model(p, compile=False)

#     # Sidecar paths
#     classes_path = p.with_suffix(".classes.json")
#     meta_path    = p.with_suffix(".meta.json")

#     # --- Classes: try sidecar, else infer count and synthesize names ---
#     if classes_path.exists():
#         try:
#             CLASS_NAMES = json.loads(classes_path.read_text())
#         except Exception as e:
#             print(f"Warning: couldn't parse {classes_path.name}: {e}")
#             CLASS_NAMES = [f"class_{i}" for i in range(model.output_shape[-1])]
#     else:
#         CLASS_NAMES = [f"class_{i}" for i in range(model.output_shape[-1])]

#     # --- Meta: img_size, last_conv, etc. (optional) ---
#     meta = {}
#     if meta_path.exists():
#         try:
#             meta = json.loads(meta_path.read_text())
#         except Exception as e:
#             print(f"Warning: couldn't parse {meta_path.name}: {e}")

#     # Input info
#     inp      = model.inputs[0]
#     shape    = inp.shape
#     INPUT_NAME  = inp.name.split(":")[0]
#     INPUT_DTYPE = inp.dtype

#     # IMG_SIZE: prefer sidecar; else infer from input tensor
#     IMG_SIZE = int(meta.get("img_size") or (shape[1] if shape[1] is not None else 224))

#     # Find a valid 4D feature map layer if not provided
#     def _find_last_4d(m):
#         for lyr in reversed(m.layers):
#             try:
#                 s = getattr(lyr, "output_shape", None)
#                 if s is None:
#                     continue
#                 if isinstance(s, (list, tuple)) and s and isinstance(s[0], tuple):
#                     s = s[0]
#                 if hasattr(s, "__len__") and len(s) == 4:
#                     return lyr.name
#             except Exception:
#                 pass
#         raise ValueError("No 4D feature map layer found; cannot run Grad-CAM.")
#     LAST_CONV = meta.get("last_conv") or _find_last_4d(model)

#     # Diagnostics
#     print(f"Loaded: {p.name}")
#     print(f"Input -> name='{INPUT_NAME}', dtype={INPUT_DTYPE.name}, shape={tuple(shape)}")
#     print(f"IMG_SIZE={IMG_SIZE}  |  LAST_CONV='{LAST_CONV}'  |  #classes={len(CLASS_NAMES)}")

#     # NOTE: Preprocessing is assumed to be embedded in the model graph.
#     # Do NOT apply external preprocess_input in this notebook.

#     return model, CLASS_NAMES, IMG_SIZE, INPUT_NAME, INPUT_DTYPE, LAST_CONV

# # ---- call it ----
# MODEL_PATH = "models/resnet50_profilepic_classifier.keras"   # adjust if needed
# model, CLASS_NAMES, IMG_SIZE, INPUT_NAME, INPUT_DTYPE, LAST_CONV = load_model_and_meta(MODEL_PATH)

## Extra Stuff that Probably will be Deleted

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
# List the contents of the specified directory in Google Drive
# !ls /content/drive/MyDrive/"Colab Data"/Capstone

In [ ]:
# def load_files():
#   import os
#   from pathlib import Path

#   # Define source and destination paths
#   source_path = Path('/content/drive/MyDrive/Colab Data/Capstone/resnet50_profilepic_classifier.keras')
#   destination_dir = Path('models/baseline_savedmodel')
#   destination_path = destination_dir / source_path.name

#   # Create the destination directory if it doesn't exist
#   destination_dir.mkdir(parents=True, exist_ok=True)

#   # Copy the file using shell command
#   !cp "{source_path}" "{destination_path}"

#   print(f"Copied {source_path} to {destination_path}")

# load_files()

# Task
Create a GradCAM notebook that loads a Keras model from the "along-capstone-data" GCS bucket, specifically from the "models" folder, and uses it to generate GradCAM visualizations for images in the "along-capstone-data" GCS bucket, specifically from the "data" folder. The notebook should include steps for loading the model and data, preprocessing the images, generating predictions, calculating GradCAM heatmaps, and visualizing the results. The notebook should also include error handling for model and data loading, and for building the dataset index. The notebook should also include timing for the data loading operation.

## Setup

### Subtask:
Install necessary libraries.


**Reasoning**:
The subtask is to install the necessary libraries. The provided notebook already contains a code cell with the required `pip install` command. I will execute this cell to install the libraries.



In [29]:
!pip install tensorflow pillow opencv-python-headless numpy pandas matplotlib tqdm scikit-learn
from pathlib import Path

# ===== USER TUNABLES =====
# MODEL_PATH = Path("models/baseline_savedmodel/resnet50_profilepic_classifier.keras")  # dir or file (.keras / .h5 / SavedModel dir)
MODEL_PATH = Path("models/resnet50_profilepic_classifier.keras")
CLASS_NAMES = None

DATA_ROOT = Path("final")
METADATA_CSV = None

TARGET_SPLIT = "val"
IMG_SIZE = 224
BATCH = 32

PREPROCESS = "resnet50"  # 'resnet50' | 'efficientnet' | 'none'
TARGET_LAYER_NAME = None # e.g., "conv5_block3_out"; None => auto-detect last conv
ALPHA = 0.35

N_MISCLASS_PER_CLASS = 8
N_CORRECT_PER_CLASS = 6

OUT_DIR = Path("gradcam_outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)
# =========================

## Load model and metadata

### Subtask:
Load the Keras model and associated metadata (like class names, image size, and the target convolutional layer name) from the specified paths.


**Reasoning**:
Define the function `load_model_and_meta` to load the model and its associated metadata as per the instructions.



In [30]:
# === GradcamInspection: DROP-IN REPLACEMENT for model + metadata loading ===
from pathlib import Path
from keras.models import load_model
import tensorflow as tf
import json

def load_model_and_meta(model_path: str | Path):
    p = Path(model_path)
    if not p.exists():
        raise FileNotFoundError(f"Model file not found: {p.resolve()}")

    # Load the Keras single-file model (no compile needed for inference)
    model = load_model(p, compile=False)

    # Sidecar paths
    classes_path = p.with_suffix(".classes.json")
    meta_path    = p.with_suffix(".meta.json")

    # --- Classes: try sidecar, else infer count and synthesize names ---
    if classes_path.exists():
        try:
            CLASS_NAMES_local = json.loads(classes_path.read_text())
        except Exception as e:
            print(f"Warning: couldn't parse {classes_path.name}: {e}")
            CLASS_NAMES_local = [f"class_{i}" for i in range(model.output_shape[-1])]
    else:
        CLASS_NAMES_local = [f"class_{i}" for i in range(model.output_shape[-1])]

    # --- Meta: img_size, last_conv, etc. (optional) ---
    meta_local = {}
    if meta_path.exists():
        try:
            meta_local = json.loads(meta_path.read_text())
        except Exception as e:
            print(f"Warning: couldn't parse {meta_path.name}: {e}")

    # Input info
    inp      = model.inputs[0]
    shape    = inp.shape
    INPUT_NAME_local  = inp.name.split(":")[0]
    INPUT_DTYPE_local = inp.dtype

    # IMG_SIZE: prefer sidecar; else infer from input tensor
    IMG_SIZE_local = int(meta_local.get("img_size") or (shape[1] if shape[1] is not None else 224))

    # Find a valid 4D feature map layer if not provided
    def _find_last_4d(m):
        for lyr in reversed(m.layers):
            try:
                s = getattr(lyr, "output_shape", None)
                if s is None:
                    continue
                if isinstance(s, (list, tuple)) and s and isinstance(s[0], tuple):
                    s = s[0]
                if hasattr(s, "__len__") and len(s) == 4:
                    return lyr.name
            except Exception:
                pass
        raise ValueError("No 4D feature map layer found; cannot run Grad-CAM.")
    LAST_CONV_local = meta_local.get("last_conv") or _find_last_4d(model)

    # Diagnostics
    print(f"Loaded: {p.name}")
    print(f"Input -> name='{INPUT_NAME_local}', dtype={INPUT_DTYPE_local.name}, shape={tuple(shape)}")
    print(f"IMG_SIZE={IMG_SIZE_local}  |  LAST_CONV='{LAST_CONV_local}'  |  #classes={len(CLASS_NAMES_local)}")

    # NOTE: Preprocessing is assumed to be embedded in the model graph.
    # Do NOT apply external preprocess_input in this notebook.

    return model, CLASS_NAMES_local, IMG_SIZE_local, INPUT_NAME_local, INPUT_DTYPE_local, LAST_CONV_local

# ---- call it ----
MODEL_PATH = "models/resnet50_profilepic_classifier.keras"   # adjust if needed
model, CLASS_NAMES, IMG_SIZE, INPUT_NAME, INPUT_DTYPE, LAST_CONV = load_model_and_meta(MODEL_PATH)


Loaded: resnet50_profilepic_classifier.keras


AttributeError: 'str' object has no attribute 'name'

**Reasoning**:
The previous code failed because `INPUT_DTYPE_local` is a string and does not have a `.name` attribute. The print statement needs to be fixed to directly print `INPUT_DTYPE_local`. Also, the global variables need to be updated within the function scope or returned and assigned outside. I will return the values and assign them outside the function.



In [31]:
# === GradcamInspection: DROP-IN REPLACEMENT for model + metadata loading ===
from pathlib import Path
from keras.models import load_model
import tensorflow as tf
import json

def load_model_and_meta(model_path: str | Path):
    p = Path(model_path)
    if not p.exists():
        raise FileNotFoundError(f"Model file not found: {p.resolve()}")

    # Load the Keras single-file model (no compile needed for inference)
    model = load_model(p, compile=False)

    # Sidecar paths
    classes_path = p.with_suffix(".classes.json")
    meta_path    = p.with_suffix(".meta.json")

    # --- Classes: try sidecar, else infer count and synthesize names ---
    if classes_path.exists():
        try:
            CLASS_NAMES_local = json.loads(classes_path.read_text())
        except Exception as e:
            print(f"Warning: couldn't parse {classes_path.name}: {e}")
            CLASS_NAMES_local = [f"class_{i}" for i in range(model.output_shape[-1])]
    else:
        CLASS_NAMES_local = [f"class_{i}" for i in range(model.output_shape[-1])]

    # --- Meta: img_size, last_conv, etc. (optional) ---
    meta_local = {}
    if meta_path.exists():
        try:
            meta_local = json.loads(meta_path.read_text())
        except Exception as e:
            print(f"Warning: couldn't parse {meta_path.name}: {e}")

    # Input info
    inp      = model.inputs[0]
    shape    = inp.shape
    INPUT_NAME_local  = inp.name.split(":")[0]
    INPUT_DTYPE_local = inp.dtype

    # IMG_SIZE: prefer sidecar; else infer from input tensor
    IMG_SIZE_local = int(meta_local.get("img_size") or (shape[1] if shape[1] is not None else 224))

    # Find a valid 4D feature map layer if not provided
    def _find_last_4d(m):
        for lyr in reversed(m.layers):
            try:
                s = getattr(lyr, "output_shape", None)
                if s is None:
                    continue
                if isinstance(s, (list, tuple)) and s and isinstance(s[0], tuple):
                    s = s[0]
                if hasattr(s, "__len__") and len(s) == 4:
                    return lyr.name
            except Exception:
                pass
        raise ValueError("No 4D feature map layer found; cannot run Grad-CAM.")
    LAST_CONV_local = meta_local.get("last_conv") or _find_last_4d(model)

    # Diagnostics
    print(f"Loaded: {p.name}")
    print(f"Input -> name='{INPUT_NAME_local}', dtype={INPUT_DTYPE_local}, shape={tuple(shape)}")
    print(f"IMG_SIZE={IMG_SIZE_local}  |  LAST_CONV='{LAST_CONV_local}'  |  #classes={len(CLASS_NAMES_local)}")

    # NOTE: Preprocessing is assumed to be embedded in the model graph.
    # Do NOT apply external preprocess_input in this notebook.

    return model, CLASS_NAMES_local, IMG_SIZE_local, INPUT_NAME_local, INPUT_DTYPE_local, LAST_CONV_local

# ---- call it ----
MODEL_PATH = "models/resnet50_profilepic_classifier.keras"   # adjust if needed
model, CLASS_NAMES, IMG_SIZE, INPUT_NAME, INPUT_DTYPE, LAST_CONV = load_model_and_meta(MODEL_PATH)

Loaded: resnet50_profilepic_classifier.keras
Input -> name='image', dtype=float32, shape=(None, 224, 224, 3)
IMG_SIZE=224  |  LAST_CONV='conv5_block3_out'  |  #classes=3


## Build dataset index

### Subtask:
Build a DataFrame indexing the images and their labels from the data root directory for the target split.


**Reasoning**:
Implement the logic to build the dataset index from the data root directory, filter it for the target split, handle empty dataframes, define class names if not provided, and create label-to-index and index-to-label mappings as per the instructions.



In [32]:
def list_images_directory(root: Path, split: str) -> pd.DataFrame:
    print(f"Checking directory: {root / split}")
    rows = []
    base = root / split
    if not base.exists():
        print(f"Directory does not exist: {base}")
        return pd.DataFrame(columns=["path","label","split"])
    for cls_dir in sorted([p for p in base.iterdir() if p.is_dir()]):
        for img in cls_dir.rglob("*"):
            if img.suffix.lower() in {".jpg",".jpeg",".png",".bmp",".webp"}:
                rows.append({"path": str(img.as_posix()), "label": cls_dir.name, "split": split})
    return pd.DataFrame(rows)

if METADATA_CSV:
    df_all = pd.read_csv(METADATA_CSV)
    need = {"path","label","split"}
    if not need.issubset(set(df_all.columns)):
        raise ValueError(f"CSV must contain columns: {need}")
    df_all["path"] = df_all["path"].astype(str)
else:
    df_train = list_images_directory(DATA_ROOT, "train")
    df_val   = list_images_directory(DATA_ROOT, "val")
    df_test  = list_images_directory(DATA_ROOT, "test")
    df_all = pd.concat([df_train, df_val, df_test], ignore_index=True)
    print(f"head: {df_all.head()}")
    print(f"shape:{df_all.shape}")

df = df_all[df_all["split"] == TARGET_SPLIT].copy().reset_index(drop=True)

if df.empty:
    raise RuntimeError(f"Failed to build dataset index for split '{TARGET_SPLIT}'. No data found.")

if CLASS_NAMES is None:
    CLASS_NAMES = sorted(df_all["label"].dropna().unique().tolist())

label_to_index = {c:i for i,c in enumerate(CLASS_NAMES)}
index_to_label = {i:c for c,i in label_to_index.items()}

print("Classes:", CLASS_NAMES)
print("Counts:", df["label"].value_counts())

Checking directory: final/train
Checking directory: final/val
Checking directory: final/test
head:                                       path   label  split
0  final/train/animal/93ec4b691675b6c0.jpg  animal  train
1  final/train/animal/94421ccfa93847d3.jpg  animal  train
2  final/train/animal/d8be689340c99353.jpg  animal  train
3  final/train/animal/a06412878921e7c6.jpg  animal  train
4  final/train/animal/fba40c566603c69b.jpg  animal  train
shape:(29872, 3)
Classes: ['human', 'avatar', 'animal']
Counts: label
animal    1000
avatar    1000
human      987
Name: count, dtype: int64


## Data pipeline and preprocessing

### Subtask:
Set up a `tf.data` pipeline to load and preprocess images, ensuring the preprocessing matches the model's requirements and handles potential input structure variations (e.g., named inputs).


**Reasoning**:
Define the `preprocess_image` and `build_ds` functions to create the tf.data pipeline for the evaluation dataset.



In [33]:
def preprocess_image(path: tf.Tensor) -> tf.Tensor:
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE), method=tf.image.ResizeMethod.BILINEAR)
    img = tf.cast(img, tf.float32)
    if PREPROCESS.lower() == "resnet50":
        from keras.applications.resnet50 import preprocess_input
        img = preprocess_input(img)
    elif PREPROCESS.lower() == "efficientnet":
        from keras.applications.efficientnet import preprocess_input
        img = efficientnet.preprocess_input(img)
    else:
        img = img / 255.0
    return img

def build_ds(paths: List[str], labels: List[int], batch=BATCH, shuffle=False) -> tf.data.Dataset:
    x = tf.constant(paths, dtype=tf.string)
    y = tf.constant(labels, dtype=tf.int32)
    ds = tf.data.Dataset.from_tensor_slices((x,y))
    if shuffle:
        ds = ds.shuffle(len(paths), reshuffle_each_iteration=False)
    ds = ds.map(lambda p,l: (preprocess_image(p), tf.one_hot(l, depth=len(CLASS_NAMES))),
                num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch).prefetch(tf.data.AUTOTUNE)
    return ds

paths = df["path"].tolist()
labels = [label_to_index[l] for l in df["label"].tolist()]
ds_eval = build_ds(paths, labels, batch=BATCH, shuffle=False)

## Prediction

### Subtask:
Run inference on the dataset to get predictions and build a score table including true labels, predicted labels, confidence scores, and correctness.


**Reasoning**:
Run inference on the dataset and build the score table as instructed.



In [34]:
probs = []
for xb, yb in ds_eval:
    p = model.predict(xb, verbose=0)
    probs.append(p)
probs = np.vstack(probs)

pred_idx = probs.argmax(axis=1)
pred_lbl = [index_to_label[i] for i in pred_idx]
true_lbl = df["label"].tolist()

conf = probs[np.arange(len(probs)), pred_idx]

score_df = pd.DataFrame({
    "path": paths,
    "true_label": true_lbl,
    "pred_label": pred_lbl,
    "pred_idx": pred_idx,
    "true_idx": [label_to_index[t] for t in true_lbl],
    "confidence": conf
})
score_df["is_correct"] = score_df["true_label"] == score_df["pred_label"]

display(score_df.head())

,path,true_label,pred_label,pred_idx,true_idx,confidence,is_correct
0,final/val/animal/df7b903a20dcb96f.jpg,animal,animal,2,2,1.0,True
1,final/val/animal/d32586fd66f8e342.jpg,animal,animal,2,2,1.0,True
2,final/val/animal/844ac04c499e79c3.jpg,animal,animal,2,2,1.0,True
3,final/val/animal/9744f159355e54f2.jpg,animal,animal,2,2,1.0,True
4,final/val/animal/f50b6f9bbbf4a9c0.jpg,animal,animal,2,2,1.0,True


## Grad-cam utilities

### Subtask:
Implement functions for calculating Grad-CAM heatmaps and overlaying them on images, including handling the complexities of accessing intermediate layers and computing gradients with the loaded model. Also include a utility for calculating border attention.


**Reasoning**:
Implement the utility functions for Grad-CAM visualization as described in the instructions, removing any redundant code.



In [35]:
import cv2
import numpy as np
import tensorflow as tf
from PIL import Image
from keras import Model

def find_last_conv_layer(model: tf.keras.Model) -> str:
    """Finds the name of the last 4D convolutional layer in the model."""
    for layer in reversed(model.layers):
        try:
            out_shape = layer.output_shape
            if isinstance(out_shape, list):
                out_shape = out_shape[0]
            if len(out_shape) == 4:
                print(f"Found last 4D layer: {layer.name}")
                return layer.name
        except Exception:
            continue
    raise ValueError("No 4D conv layer found; set TARGET_LAYER_NAME manually.")

def make_gradcam_heatmap(
    x: tf.Tensor,
    class_index: int | None = None,
    last_conv_layer_name: str | None = None,
):
    """Generates a Grad-CAM heatmap for a given image tensor."""
    if last_conv_layer_name is None:
        last_conv_layer_name = find_last_conv_layer(model)

    conv_layer = model.get_layer(last_conv_layer_name)

    # Build a grad model that preserves the original input structure
    grad_model = Model(inputs=model.inputs, outputs=[conv_layer.output, model.output])

    with tf.GradientTape() as tape:
        tape.watch(x)  # make sure gradients can flow wrt the input image
        # Use the determined CALL_STYLE to call the grad_model
        conv_outputs, preds = call_like(grad_model, x)

        if class_index is None:
            # Assuming preds is a tensor (batch_size, num_classes) or similar structure
            # If model output is more complex, this might need adjustment
            if isinstance(preds, (list, tuple)):
                 # Assuming the first element is the primary output
                 preds_tensor = preds[0]
            else:
                 preds_tensor = preds

            if preds_tensor.shape.rank == 2:
                class_index = int(tf.argmax(preds_tensor[0]))
            else:
                 # Handle other potential output shapes if necessary
                 raise ValueError(f"Model output shape not supported for auto class index: {preds_tensor.shape}")


        # Calculate the target score
        target = preds_tensor[:, class_index]


    grads = tape.gradient(target, conv_outputs)               # (1, H, W, C)
    pooled = tf.reduce_mean(grads, axis=(0, 1, 2))            # (C,)
    fmap  = conv_outputs[0]                                   # (H, W, C)
    cam   = tf.reduce_sum(fmap * pooled, axis=-1)             # (H, W)

    cam = tf.maximum(cam, 0.0)
    mx  = tf.reduce_max(cam)
    cam = tf.where(mx > 0, cam / mx, tf.zeros_like(cam))
    return cam.numpy()

def overlay_heatmap_on_image(path: str, heatmap: np.ndarray, alpha=ALPHA, img_size=IMG_SIZE):
    """Overlays a heatmap onto the original image."""
    img = Image.open(path).convert("RGB").resize((img_size, img_size))
    img_np = np.array(img)
    hm = cv2.resize(heatmap, (img_size, img_size))
    hm = np.uint8(255 * hm)
    hm_color = cv2.applyColorMap(hm, cv2.COLORMAP_JET)
    # Ensure dimensions match for cv2.addWeighted
    if img_np.shape[-1] == 3 and hm_color.shape[-1] == 3:
         overlay = cv2.addWeighted(hm_color, alpha, img_np[:, :, ::-1], 1.0 - alpha, 0)
    else:
         # Handle cases where image might be grayscale or heatmap is not 3 channels
         # This simplified version assumes both are RGB or compatible
         print("Warning: Image or heatmap not in expected format for overlay. Skipping overlay.")
         return img_np, img_np # Return original image for both

    overlay = overlay[:, :, ::-1] # Convert BGR to RGB
    return img_np, overlay

def border_attention_fraction(heatmap: np.ndarray, border_ratio: float = 0.08) -> float:
    """Calculates the fraction of attention in the border region of a heatmap."""
    h, w = heatmap.shape
    b = int(round(min(h, w) * border_ratio))
    core = heatmap[b:h-b, b:w-b].sum() if (b > 0 and (h - 2 * b) > 0 and (w - 2 * b) > 0) else 0.0
    total = heatmap.sum() + 1e-8
    border = total - core
    return float(border / total)

# Remove redundant function definitions
del find_last_conv_layer
del make_gradcam_heatmap
del overlay_heatmap_on_image
del border_attention_fraction

# Redefine the functions with the correct implementations
def find_last_conv_layer(model: tf.keras.Model) -> str:
    """Finds the name of the last 4D convolutional layer in the model."""
    for layer in reversed(model.layers):
        try:
            out_shape = layer.output_shape
            if isinstance(out_shape, list):
                out_shape = out_shape[0]
            if len(out_shape) == 4:
                print(f"Found last 4D layer: {layer.name}")
                return layer.name
        except Exception:
            continue
    raise ValueError("No 4D conv layer found; set TARGET_LAYER_NAME manually.")

def make_gradcam_heatmap(
    x: tf.Tensor,
    class_index: int | None = None,
    last_conv_layer_name: str | None = None,
):
    """Generates a Grad-CAM heatmap for a given image tensor."""
    if last_conv_layer_name is None:
        last_conv_layer_name = find_last_conv_layer(model)

    conv_layer = model.get_layer(last_conv_layer_name)

    # Build a grad model that preserves the original input structure
    grad_model = Model(inputs=model.inputs, outputs=[conv_layer.output, model.output])

    with tf.GradientTape() as tape:
        tape.watch(x)  # make sure gradients can flow wrt the input image
        # Use the determined CALL_STYLE to call the grad_model
        conv_outputs, preds = call_like(grad_model, x)

        if class_index is None:
            # Assuming preds is a tensor (batch_size, num_classes) or similar structure
            # If model output is more complex, this might need adjustment
            if isinstance(preds, (list, tuple)):
                 # Assuming the first element is the primary output
                 preds_tensor = preds[0]
            else:
                 preds_tensor = preds

            if preds_tensor.shape.rank == 2:
                class_index = int(tf.argmax(preds_tensor[0]))
            else:
                 # Handle other potential output shapes if necessary
                 raise ValueError(f"Model output shape not supported for auto class index: {preds_tensor.shape}")


        # Calculate the target score
        target = preds_tensor[:, class_index]


    grads = tape.gradient(target, conv_outputs)               # (1, H, W, C)
    pooled = tf.reduce_mean(grads, axis=(0, 1, 2))            # (C,)
    fmap  = conv_outputs[0]                                   # (H, W, C)
    cam   = tf.reduce_sum(fmap * pooled, axis=-1)             # (H, W)

    cam = tf.maximum(cam, 0.0)
    mx  = tf.reduce_max(cam)
    cam = tf.where(mx > 0, cam / mx, tf.zeros_like(cam))
    return cam.numpy()

def overlay_heatmap_on_image(path: str, heatmap: np.ndarray, alpha=ALPHA, img_size=IMG_SIZE):
    """Overlays a heatmap onto the original image."""
    img = Image.open(path).convert("RGB").resize((img_size, img_size))
    img_np = np.array(img)
    hm = cv2.resize(heatmap, (img_size, img_size))
    hm = np.uint8(255 * hm)
    hm_color = cv2.applyColorMap(hm, cv2.COLORMAP_JET)
    # Ensure dimensions match for cv2.addWeighted
    if img_np.shape[-1] == 3 and hm_color.shape[-1] == 3:
         overlay = cv2.addWeighted(hm_color, alpha, img_np[:, :, ::-1], 1.0 - alpha, 0)
    else:
         # Handle cases where image might be grayscale or heatmap is not 3 channels
         # This simplified version assumes both are RGB or compatible
         print("Warning: Image or heatmap not in expected format for overlay. Skipping overlay.")
         return img_np, img_np # Return original image for both

    overlay = overlay[:, :, ::-1] # Convert BGR to RGB
    return img_np, overlay

def border_attention_fraction(heatmap: np.ndarray, border_ratio: float = 0.08) -> float:
    """Calculates the fraction of attention in the border region of a heatmap."""
    h, w = heatmap.shape
    b = int(round(min(h, w) * border_ratio))
    core = heatmap[b:h-b, b:w-b].sum() if (b > 0 and (h - 2 * b) > 0 and (w - 2 * b) > 0) else 0.0
    total = heatmap.sum() + 1e-8
    border = total - core
    return float(border / total)

## Visualize grad-cam panels

### Subtask:
Select samples (correctly classified and misclassified) for each class and generate/save visualizations showing the original image, heatmap, and overlay with border attention score.


**Reasoning**:
Define the functions for picking samples and generating panels, then use them to generate and save the Grad-CAM visualizations for correct and misclassified examples for each class.



In [36]:
import math
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from PIL import Image
import cv2
import tensorflow as tf
from pathlib import Path
from typing import List, Tuple

def pick_samples(score_df: pd.DataFrame, per_class_mis:int, per_class_ok:int):
    """Selects correctly classified and misclassified images per class."""
    rows = []
    for cls in CLASS_NAMES:
        sub = score_df[score_df["true_label"] == cls].copy()
        # Sort by confidence to get the "most" misclassified/correct
        mis = sub[~sub["is_correct"]].sort_values("confidence", ascending=False).head(per_class_mis)
        ok  = sub[sub["is_correct"]].sort_values("confidence", ascending=False).head(per_class_ok)
        rows.append(("MIS", cls, mis))
        rows.append(("OK",  cls, ok))
    return rows

def panel_for_group(kind: str, cls: str, group_df: pd.DataFrame, save_path: Path):
    """Generates and saves a panel of visualizations for a group of images."""
    n = len(group_df)
    if n == 0:
        print(f"No samples for {kind}: {cls}")
        return

    cols = 3 # Original, Heatmap, Overlay
    rows = math.ceil(n) # Each sample gets one row
    fig_h = max(4, rows * 3) # Adjust figure height based on number of rows
    fig_w = 12

    plt.figure(figsize=(fig_w, fig_h))
    idx = 1
    records = []

    for r in group_df.itertuples(index=False):
        try:
            x = preprocess_for_single(r.path)
            heat = make_gradcam_heatmap(x, class_index=r.pred_idx)
            img_np, overlay = overlay_heatmap_on_image(r.path, heat, alpha=ALPHA, img_size=IMG_SIZE)
            frac = border_attention_fraction(heat)

            # Original Image
            plt.subplot(rows, cols, idx);
            plt.imshow(img_np);
            plt.axis("off");
            plt.title(f"Orig\ntrue={r.true_label}\npred={r.pred_label}\nconf={r.confidence:.2f}")
            idx += 1

            # Heatmap
            plt.subplot(rows, cols, idx);
            plt.imshow(heat, cmap="jet");
            plt.axis("off");
            plt.title("Heatmap")
            idx += 1

            # Overlay
            plt.subplot(rows, cols, idx);
            plt.imshow(overlay);
            plt.axis("off");
            plt.title(f"Overlay\nborder={frac:.2f}")
            idx += 1

            records.append({
                "path": r.path,
                "true_label": r.true_label,
                "pred_label": r.pred_label,
                "confidence": r.confidence,
                "border_attention_frac": frac,
                "is_correct": r.is_correct
            })
        except Exception as e:
            print(f"Error processing image {r.path}: {e}")
            # Add a placeholder or skip the row if processing fails
            # For now, let's just print the error and continue

    if records: # Only try to save if there were successful records
        plt.suptitle(f"{kind}: {cls} — {len(records)} samples", y=1.02)
        plt.tight_layout()
        plt.savefig(save_path, dpi=160, bbox_inches="tight")
        plt.show()

        pd.DataFrame(records).to_csv(save_path.with_suffix(".csv"), index=False)
    else:
        print(f"No successful visualizations for {kind}: {cls}. Skipping save.")
    plt.close() # Close the figure to free memory


# Select samples
samples = pick_samples(score_df, N_MISCLASS_PER_CLASS, N_CORRECT_PER_CLASS)

# Generate and save panels
for kind, cls, gdf in samples:
    slug = f"{kind.lower()}_{cls}".replace(" ", "_")
    out_file = OUT_DIR / f"gradcam_{slug}.png"
    panel_for_group(kind, cls, gdf, out_file)

print("Saved panels to:", OUT_DIR.resolve())


Found last 4D layer: resnet50
Error processing image final/val/human/7a89ac6facd28bd8.jpg: name 'call_like' is not defined
Found last 4D layer: resnet50
Error processing image final/val/human/dbef3f001468883a.jpg: name 'call_like' is not defined
Found last 4D layer: resnet50
Error processing image final/val/human/306c5a7e331eb6e4.jpg: name 'call_like' is not defined
Found last 4D layer: resnet50
Error processing image final/val/human/c1e39f353a3fa88a.jpg: name 'call_like' is not defined
Found last 4D layer: resnet50
Error processing image final/val/human/7b62774aece0c95f.jpg: name 'call_like' is not defined
Found last 4D layer: resnet50
Error processing image final/val/human/28fd4bcec39c2d14.jpg: name 'call_like' is not defined
Found last 4D layer: resnet50
Error processing image final/val/human/bb4e10e42934911f.jpg: name 'call_like' is not defined
Found last 4D layer: resnet50
Error processing image final/val/human/cf04834dcd10abe3.jpg: name 'call_like' is not defined
No successful vi

**Reasoning**:
The previous execution failed because the `call_like` function was not defined within the scope of the `panel_for_group` function. The `call_like` function and the updated `make_gradcam_heatmap` function that uses it need to be included in the same code block as `panel_for_group`.



In [37]:
import math
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from PIL import Image
import cv2
import tensorflow as tf
from pathlib import Path
from typing import List, Tuple
from keras import Model # Import Model from keras

def pick_samples(score_df: pd.DataFrame, per_class_mis:int, per_class_ok:int):
    """Selects correctly classified and misclassified images per class."""
    rows = []
    for cls in CLASS_NAMES:
        sub = score_df[score_df["true_label"] == cls].copy()
        # Sort by confidence to get the "most" misclassified/correct
        mis = sub[~sub["is_correct"]].sort_values("confidence", ascending=False).head(per_class_mis)
        ok  = sub[sub["is_correct"]].sort_values("confidence", ascending=False).head(per_class_ok)
        rows.append(("MIS", cls, mis))
        rows.append(("OK",  cls, ok))
    return rows

# Define call_like function again so it's in scope
def call_like(m, x):
    """
    Try calling model with bare tensor, [tensor], and {input_name: tensor}.
    Works around Keras 3 structured-input expectations (e.g., Input(name="image", ...)).
    """
    # 1) Try bare tensor
    try:
        return m(x, training=False)
    except Exception:
        pass
    # 2) Try list-wrapped (single-input models often accept [x])
    try:
        return m([x], training=False)
    except Exception:
        pass
    # 3) Try dict by first input name (assuming INPUT_KEY is defined)
    try:
        if 'INPUT_KEY' in globals() and INPUT_KEY is not None:
             return m({INPUT_KEY: x}, training=False)
    except Exception:
        pass
    # If still failing, raise a clear error
    # Check if x has a shape attribute
    x_shape = getattr(x, 'shape', 'N/A')
    input_names = getattr(m, 'input_names', None)

    raise RuntimeError(
        f"Could not call model with structured inputs. "
        f"Input names={input_names}; got tensor shape={x_shape}"
    )


# Redefine make_gradcam_heatmap to use call_like
def make_gradcam_heatmap(
    x: tf.Tensor,
    class_index: int | None = None,
    last_conv_layer_name: str | None = None,
):
    """Generates a Grad-CAM heatmap for a given image tensor."""
    if last_conv_layer_name is None:
        last_conv_layer_name = find_last_conv_layer(model)

    conv_layer = model.get_layer(last_conv_layer_name)

    # Build a grad model that preserves the original input structure
    grad_model = Model(inputs=model.inputs, outputs=[conv_layer.output, model.output])

    with tf.GradientTape() as tape:
        tape.watch(x)  # make sure gradients can flow wrt the input image
        # <<< KEY CHANGE: call the grad_model using call_like >>>
        conv_outputs, preds = call_like(grad_model, x)

        if class_index is None:
            # Assuming preds is a tensor (batch_size, num_classes) or similar structure
            # If model output is more complex, this might need adjustment
            if isinstance(preds, (list, tuple)):
                 # Assuming the first element is the primary output
                 preds_tensor = preds[0]
            else:
                 preds_tensor = preds

            if preds_tensor.shape.rank == 2:
                class_index = int(tf.argmax(preds_tensor[0]))
            else:
                 # Handle other potential output shapes if necessary
                 raise ValueError(f"Model output shape not supported for auto class index: {preds_tensor.shape}")


        # Calculate the target score
        target = preds_tensor[:, class_index]


    grads = tape.gradient(target, conv_outputs)               # (1, H, W, C)
    pooled = tf.reduce_mean(grads, axis=(0, 1, 2))            # (C,)
    fmap  = conv_outputs[0]                                   # (H, W, C)
    cam   = tf.reduce_sum(fmap * pooled, axis=-1)             # (H, W)

    cam = tf.maximum(cam, 0.0)
    mx  = tf.reduce_max(cam)
    cam = tf.where(mx > 0, cam / mx, tf.zeros_like(cam))
    return cam.numpy()


# Redefine overlay_heatmap_on_image
def overlay_heatmap_on_image(path: str, heatmap: np.ndarray, alpha=ALPHA, img_size=IMG_SIZE):
    """Overlays a heatmap onto the original image."""
    img = Image.open(path).convert("RGB").resize((img_size, img_size))
    img_np = np.array(img)
    hm = cv2.resize(heatmap, (img_size, img_size))
    hm = np.uint8(255 * hm)
    hm_color = cv2.applyColorMap(hm, cv2.COLORMAP_JET)
    # Ensure dimensions match for cv2.addWeighted
    if img_np.shape[-1] == 3 and hm_color.shape[-1] == 3:
         overlay = cv2.addWeighted(hm_color, alpha, img_np[:, :, ::-1], 1.0 - alpha, 0)
    else:
         # Handle cases where image might be grayscale or heatmap is not 3 channels
         # This simplified version assumes both are RGB or compatible
         print("Warning: Image or heatmap not in expected format for overlay. Skipping overlay.")
         return img_np, img_np # Return original image for both

    overlay = overlay[:, :, ::-1] # Convert BGR to RGB
    return img_np, overlay

# Redefine border_attention_fraction
def border_attention_fraction(heatmap: np.ndarray, border_ratio: float = 0.08) -> float:
    """Calculates the fraction of attention in the border region of a heatmap."""
    h, w = heatmap.shape
    b = int(round(min(h, w) * border_ratio))
    core = heatmap[b:h-b, b:w-b].sum() if (b > 0 and (h - 2 * b) > 0 and (w - 2 * b) > 0) else 0.0
    total = heatmap.sum() + 1e-8
    border = total - core
    return float(border / total)

# Redefine find_last_conv_layer
def find_last_conv_layer(model: tf.keras.Model) -> str:
    """Finds the name of the last 4D convolutional layer in the model."""
    for layer in reversed(model.layers):
        try:
            out_shape = layer.output_shape
            if isinstance(out_shape, list):
                out_shape = out_shape[0]
            if len(out_shape) == 4:
                print(f"Found last 4D layer: {layer.name}")
                return layer.name
        except Exception:
            continue
    raise ValueError("No 4D conv layer found; set TARGET_LAYER_NAME manually.")


def panel_for_group(kind: str, cls: str, group_df: pd.DataFrame, save_path: Path):
    """Generates and saves a panel of visualizations for a group of images."""
    n = len(group_df)
    if n == 0:
        print(f"No samples for {kind}: {cls}")
        return

    cols = 3 # Original, Heatmap, Overlay
    rows = math.ceil(n) # Each sample gets one row
    fig_h = max(4, rows * 3) # Adjust figure height based on number of rows
    fig_w = 12

    plt.figure(figsize=(fig_w, fig_h))
    idx = 1
    records = []

    for r in group_df.itertuples(index=False):
        try:
            x = preprocess_for_single(r.path)
            heat = make_gradcam_heatmap(x, class_index=r.pred_idx)
            img_np, overlay = overlay_heatmap_on_image(r.path, heat, alpha=ALPHA, img_size=IMG_SIZE)
            frac = border_attention_fraction(heat)

            # Original Image
            plt.subplot(rows, cols, idx);
            plt.imshow(img_np);
            plt.axis("off");
            plt.title(f"Orig\ntrue={r.true_label}\npred={r.pred_label}\nconf={r.confidence:.2f}")
            idx += 1

            # Heatmap
            plt.subplot(rows, cols, idx);
            plt.imshow(heat, cmap="jet");
            plt.axis("off");
            plt.title("Heatmap")
            idx += 1

            # Overlay
            plt.subplot(rows, cols, idx);
            plt.imshow(overlay);
            plt.axis("off");
            plt.title(f"Overlay\nborder={frac:.2f}")
            idx += 1

            records.append({
                "path": r.path,
                "true_label": r.true_label,
                "pred_label": r.pred_label,
                "confidence": r.confidence,
                "border_attention_frac": frac,
                "is_correct": r.is_correct
            })
        except Exception as e:
            print(f"Error processing image {r.path}: {e}")
            # Add a placeholder or skip the row if processing fails
            # For now, let's just print the error and continue

    if records: # Only try to save if there were successful records
        plt.suptitle(f"{kind}: {cls} — {len(records)} samples", y=1.02)
        plt.tight_layout()
        plt.savefig(save_path, dpi=160, bbox_inches="tight")
        plt.show()

        pd.DataFrame(records).to_csv(save_path.with_suffix(".csv"), index=False)
    else:
        print(f"No successful visualizations for {kind}: {cls}. Skipping save.")
    plt.close() # Close the figure to free memory


# Select samples
samples = pick_samples(score_df, N_MISCLASS_PER_CLASS, N_CORRECT_PER_CLASS)

# Generate and save panels
for kind, cls, gdf in samples:
    slug = f"{kind.lower()}_{cls}".replace(" ", "_")
    out_file = OUT_DIR / f"gradcam_{slug}.png"
    panel_for_group(kind, cls, gdf, out_file)

print("Saved panels to:", OUT_DIR.resolve())

Found last 4D layer: resnet50
Error processing image final/val/human/7a89ac6facd28bd8.jpg: Could not call model with structured inputs. Input names=None; got tensor shape=(1, 224, 224, 3)
Found last 4D layer: resnet50
Error processing image final/val/human/dbef3f001468883a.jpg: Could not call model with structured inputs. Input names=None; got tensor shape=(1, 224, 224, 3)
Found last 4D layer: resnet50
Error processing image final/val/human/306c5a7e331eb6e4.jpg: Could not call model with structured inputs. Input names=None; got tensor shape=(1, 224, 224, 3)
Found last 4D layer: resnet50
Error processing image final/val/human/c1e39f353a3fa88a.jpg: Could not call model with structured inputs. Input names=None; got tensor shape=(1, 224, 224, 3)
Found last 4D layer: resnet50
Error processing image final/val/human/7b62774aece0c95f.jpg: Could not call model with structured inputs. Input names=None; got tensor shape=(1, 224, 224, 3)
Found last 4D layer: resnet50
Error processing image final/v

**Reasoning**:
The error message "Could not call model with structured inputs. Input names=None; got tensor shape=(1, 224, 224, 3)" indicates that the `call_like` function is failing to call the model correctly. This might be because the loaded Keras model doesn't have explicit input names or expects a different input structure when called with a dictionary or list. The previous attempts to define `call_like` and `make_gradcam_heatmap` in separate blocks also caused issues with function scope. The most robust approach is to redefine `call_like` and `make_gradcam_heatmap` in the same block where they are used within `panel_for_group`, ensuring they have access to the necessary variables and are correctly implemented to handle the model's input expectations. I will also simplify `call_like` to primarily rely on the plain tensor call and the dictionary call using `INPUT_KEY` as these were shown to work in previous probes, and add more detailed error reporting if those fail. I will also ensure `find_last_conv_layer`, `overlay_heatmap_on_image`, and `border_attention_fraction` are defined within the same block or are accessible.



In [38]:
import math
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from PIL import Image
import cv2
import tensorflow as tf
from pathlib import Path
from typing import List, Tuple
from keras import Model # Import Model from keras

# Redefine call_like function
def call_like(m, x):
    """
    Try calling model with bare tensor or {input_name: tensor}.
    Works around Keras 3 structured-input expectations.
    """
    # 1) Try bare tensor call (most common and often works)
    try:
        return m(x, training=False)
    except Exception as e_tensor:
        # 2) Try dict by input name (if INPUT_KEY is available)
        if 'INPUT_KEY' in globals() and INPUT_KEY is not None:
            try:
                return m({INPUT_KEY: x}, training=False)
            except Exception as e_dict:
                # 3) Try list-wrapped (less common for single input but worth a shot)
                try:
                    return m([x], training=False)
                except Exception as e_list:
                         # If still failing, raise a clear error
                         x_shape = getattr(x, 'shape', 'N/A')
                         input_names = getattr(m, 'input_names', None)
                         raise RuntimeError(
                             f"Could not call model with structured inputs. "
                             f"Input names={input_names}; got tensor shape={x_shape}. "
                             f"Errors: Tensor call failed ({e_tensor}), Dict call failed ({e_dict}), List call failed ({e_list})"
                         )
        else:
            # INPUT_KEY is not defined or None, report tensor call failure
             x_shape = getattr(x, 'shape', 'N/A')
             input_names = getattr(m, 'input_names', None)
             raise RuntimeError(
                 f"Could not call model with structured inputs. INPUT_KEY is not defined or None. "
                 f"Input names={input_names}; got tensor shape={x_shape}. "
                 f"Error: Tensor call failed ({e_tensor})"
             )
    # This part should ideally not be reached if an exception is raised above
    # But as a fallback, include a generic error if none of the above worked
    x_shape = getattr(x, 'shape', 'N/A')
    input_names = getattr(m, 'input_names', None)
    raise RuntimeError(f"Unexpected error in call_like. Input names={input_names}; got tensor shape={x_shape}.")


# Redefine find_last_conv_layer
def find_last_conv_layer(model: tf.keras.Model) -> str:
    """Finds the name of the last 4D convolutional layer in the model."""
    for layer in reversed(model.layers):
        try:
            out_shape = layer.output_shape
            if isinstance(out_shape, list):
                out_shape = out_shape[0]
            if len(out_shape) == 4:
                print(f"Found last 4D layer: {layer.name}")
                return layer.name
        except Exception:
            continue
    raise ValueError("No 4D conv layer found; set TARGET_LAYER_NAME manually.")

# Redefine make_gradcam_heatmap to use call_like
def make_gradcam_heatmap(
    x: tf.Tensor,
    class_index: int | None = None,
    last_conv_layer_name: str | None = None,
):
    """Generates a Grad-CAM heatmap for a given image tensor."""
    if last_conv_layer_name is None:
        last_conv_layer_name = find_last_conv_layer(model)

    conv_layer = model.get_layer(last_conv_layer_name)

    # Build a grad model that preserves the original input structure
    grad_model = Model(inputs=model.inputs, outputs=[conv_layer.output, model.output])

    with tf.GradientTape() as tape:
        tape.watch(x)  # make sure gradients can flow wrt the input image
        # <<< KEY CHANGE: call the grad_model using call_like >>>
        conv_outputs, preds = call_like(grad_model, x)

        if class_index is None:
            # Assuming preds is a tensor (batch_size, num_classes) or similar structure
            # If model output is more complex, this might need adjustment
            if isinstance(preds, (list, tuple)):
                 # Assuming the first element is the primary output
                 preds_tensor = preds[0]
            else:
                 preds_tensor = preds

            if preds_tensor.shape.rank == 2:
                class_index = int(tf.argmax(preds_tensor[0]))
            else:
                 # Handle other potential output shapes if necessary
                 raise ValueError(f"Model output shape not supported for auto class index: {preds_tensor.shape}")


        # Calculate the target score
        target = preds_tensor[:, class_index]


    grads = tape.gradient(target, conv_outputs)               # (1, H, W, C)
    pooled = tf.reduce_mean(grads, axis=(0, 1, 2))            # (C,)
    fmap  = conv_outputs[0]                                   # (H, W, C)
    cam   = tf.reduce_sum(fmap * pooled, axis=-1)             # (H, W)

    cam = tf.maximum(cam, 0.0)
    mx  = tf.reduce_max(cam)
    cam = tf.where(mx > 0, cam / mx, tf.zeros_like(cam))
    return cam.numpy()


# Redefine overlay_heatmap_on_image
def overlay_heatmap_on_image(path: str, heatmap: np.ndarray, alpha=ALPHA, img_size=IMG_SIZE):
    """Overlays a heatmap onto the original image."""
    img = Image.open(path).convert("RGB").resize((img_size, img_size))
    img_np = np.array(img)
    hm = cv2.resize(heatmap, (img_size, img_size))
    hm = np.uint8(255 * hm)
    hm_color = cv2.applyColorMap(hm, cv2.COLORMAP_JET)
    # Ensure dimensions match for cv2.addWeighted
    if img_np.shape[-1] == 3 and hm_color.shape[-1] == 3:
         overlay = cv2.addWeighted(hm_color, alpha, img_np[:, :, ::-1], 1.0 - alpha, 0)
    else:
         # Handle cases where image might be grayscale or heatmap is not 3 channels
         # This simplified version assumes both are RGB or compatible
         print("Warning: Image or heatmap not in expected format for overlay. Skipping overlay.")
         return img_np, img_np # Return original image for both

    overlay = overlay[:, :, ::-1] # Convert BGR to RGB
    return img_np, overlay

# Redefine border_attention_fraction
def border_attention_fraction(heatmap: np.ndarray, border_ratio: float = 0.08) -> float:
    """Calculates the fraction of attention in the border region of a heatmap."""
    h, w = heatmap.shape
    b = int(round(min(h, w) * border_ratio))
    core = heatmap[b:h-b, b:w-b].sum() if (b > 0 and (h - 2 * b) > 0 and (w - 2 * b) > 0) else 0.0
    total = heatmap.sum() + 1e-8
    border = total - core
    return float(border / total)


def panel_for_group(kind: str, cls: str, group_df: pd.DataFrame, save_path: Path):
    """Generates and saves a panel of visualizations for a group of images."""
    n = len(group_df)
    if n == 0:
        print(f"No samples for {kind}: {cls}")
        return

    cols = 3 # Original, Heatmap, Overlay
    rows = math.ceil(n) # Each sample gets one row
    fig_h = max(4, rows * 3) # Adjust figure height based on number of rows
    fig_w = 12

    plt.figure(figsize=(fig_w, fig_h))
    idx = 1
    records = []

    for r in group_df.itertuples(index=False):
        try:
            x = preprocess_for_single(r.path)
            heat = make_gradcam_heatmap(x, class_index=r.pred_idx)
            img_np, overlay = overlay_heatmap_on_image(r.path, heat, alpha=ALPHA, img_size=IMG_SIZE)
            frac = border_attention_fraction(heat)

            # Original Image
            plt.subplot(rows, cols, idx);
            plt.imshow(img_np);
            plt.axis("off");
            plt.title(f"Orig\ntrue={r.true_label}\npred={r.pred_label}\nconf={r.confidence:.2f}")
            idx += 1

            # Heatmap
            plt.subplot(rows, cols, idx);
            plt.imshow(heat, cmap="jet");
            plt.axis("off");
            plt.title("Heatmap")
            idx += 1

            # Overlay
            plt.subplot(rows, cols, idx);
            plt.imshow(overlay);
            plt.axis("off");
            plt.title(f"Overlay\nborder={frac:.2f}")
            idx += 1

            records.append({
                "path": r.path,
                "true_label": r.true_label,
                "pred_label": r.pred_label,
                "confidence": r.confidence,
                "border_attention_frac": frac,
                "is_correct": r.is_correct
            })
        except Exception as e:
            print(f"Error processing image {r.path}: {e}")
            # Add a placeholder or skip the row if processing fails
            # For now, let's just print the error and continue

    if records: # Only try to save if there were successful records
        plt.suptitle(f"{kind}: {cls} — {len(records)} samples", y=1.02)
        plt.tight_layout()
        plt.savefig(save_path, dpi=160, bbox_inches="tight")
        plt.show()

        pd.DataFrame(records).to_csv(save_path.with_suffix(".csv"), index=False)
    else:
        print(f"No successful visualizations for {kind}: {cls}. Skipping save.")
    plt.close() # Close the figure to free memory


# Select samples
samples = pick_samples(score_df, N_MISCLASS_PER_CLASS, N_CORRECT_PER_CLASS)

# Generate and save panels
for kind, cls, gdf in samples:
    slug = f"{kind.lower()}_{cls}".replace(" ", "_")
    out_file = OUT_DIR / f"gradcam_{slug}.png"
    panel_for_group(kind, cls, gdf, out_file)

print("Saved panels to:", OUT_DIR.resolve())

Found last 4D layer: resnet50
Error processing image final/val/human/7a89ac6facd28bd8.jpg: Could not call model with structured inputs. Input names=None; got tensor shape=(1, 224, 224, 3). Errors: Tensor call failed ("Exception encountered when calling Functional.call().\n\n\x1b132608572170144\x1b\n\nArguments received by Functional.call():\n  • inputs=tf.Tensor(shape=(1, 224, 224, 3), dtype=float32)\n  • training=False\n  • mask=None\n  • kwargs=<class 'inspect._empty'>"), Dict call failed ("Exception encountered when calling Functional.call().\n\n\x1b132608572170144\x1b\n\nArguments received by Functional.call():\n  • inputs={'image': 'tf.Tensor(shape=(1, 224, 224, 3), dtype=float32)'}\n  • training=False\n  • mask={'image': 'None'}\n  • kwargs=<class 'inspect._empty'>"), List call failed ("Exception encountered when calling Functional.call().\n\n\x1b132608572170144\x1b\n\nArguments received by Functional.call():\n  • inputs=['tf.Tensor(shape=(1, 224, 224, 3), dtype=float32)']\n  • t

**Reasoning**:
The model loading in cell 'f5550e19' was using `keras.models.load_model` and explicitly setting `IMG_SIZE`, `CLASS_NAMES`, and `LAST_CONV` from sidecar files. This overrides the previous model loading from cell '6cd95276'. The error "Could not call model with structured inputs. Input names=None" suggests the model loaded this way might not have accessible input names or is not being called with the correct structure expected by `call_like`. Also, the `preprocess_for_single` function was redefined multiple times, potentially leading to inconsistencies. To fix this, I will: 1) Ensure only one model loading approach is used (the one from cell 'f5550e19' which loads metadata). 2) Redefine `preprocess_for_single` to use the `INPUT_DTYPE` and `IMG_SIZE` loaded with the model. 3) Simplify `call_like` to primarily rely on the plain tensor call, as the error traceback indicates that the functional model call itself is failing even with dictionary/list inputs, which might be related to the way the model was saved or loaded. 4) Redefine all necessary utility functions (`find_last_conv_layer`, `make_gradcam_heatmap`, `overlay_heatmap_on_image`, `border_attention_fraction`, `call_like`, `preprocess_for_single`, `panel_for_group`) in a single block to ensure correct scope and consistent definitions, and then run the sample picking and panel generation.



In [39]:
import math
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from PIL import Image
import cv2
import tensorflow as tf
from pathlib import Path
from typing import List, Tuple
from keras import Model # Import Model from keras

# Use the model loading and metadata from cell 'f5550e19'
# This assumes model, CLASS_NAMES, IMG_SIZE, INPUT_KEY, INPUT_DTYPE, LAST_CONV are already defined by cell 'f5550e19' or the model loading code block.

# Redefine preprocess_for_single based on loaded model metadata
def preprocess_for_single(path: str) -> tf.Tensor:
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    # Use IMG_SIZE loaded with the model
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE), method=tf.image.ResizeMethod.BILINEAR)

    # Use INPUT_DTYPE loaded with the model
    if INPUT_DTYPE == tf.uint8:
        img = tf.clip_by_value(img, 0, 255)
        img = tf.cast(img, tf.uint8)
    else:
        img = tf.cast(img, tf.float32)
        # Assuming preprocessing is handled within the loaded model graph
        # Do NOT apply external preprocess_input here if the model expects raw input

    img.set_shape([IMG_SIZE, IMG_SIZE, 3])
    x = tf.expand_dims(img, 0)
    x.set_shape([1, IMG_SIZE, IMG_SIZE, 3])
    return x


# Simplified call_like function
def call_like(m, x):
    """
    Try calling model with bare tensor.
    Fall back to dictionary call if INPUT_KEY is available.
    """
    try:
        # Try plain tensor call first
        return m(x, training=False)
    except Exception as e_tensor:
        # If tensor call fails, try dictionary call using the detected INPUT_KEY
        if 'INPUT_KEY' in globals() and INPUT_KEY is not None:
            try:
                 return m({INPUT_KEY: x}, training=False)
            except Exception as e_dict:
                 # If both fail, raise an informative error
                 x_shape = getattr(x, 'shape', 'N/A')
                 input_names = getattr(m, 'input_names', None)
                 raise RuntimeError(
                     f"Could not call model with structured inputs. "
                     f"Input names={input_names}; got tensor shape={x_shape}. "
                     f"Errors: Tensor call failed ({e_tensor}), Dict call failed ({e_dict})"
                 )
        else:
            # INPUT_KEY not available, report tensor call failure
            x_shape = getattr(x, 'shape', 'N/A')
            input_names = getattr(m, 'input_names', None)
            raise RuntimeError(
                f"Could not call model with structured inputs. INPUT_KEY is not defined or None. "
                f"Input names={input_names}; got tensor shape={x_shape}. "
                f"Error: Tensor call failed ({e_tensor})"
            )


# Redefine find_last_conv_layer
def find_last_conv_layer(model: tf.keras.Model) -> str:
    """Finds the name of the last 4D convolutional layer in the model."""
    # Use the LAST_CONV loaded from metadata if available and valid
    if 'LAST_CONV' in globals() and LAST_CONV is not None:
        try:
            layer = model.get_layer(LAST_CONV)
            out_shape = layer.output_shape
            if isinstance(out_shape, list):
                out_shape = out_shape[0]
            if len(out_shape) == 4:
                print(f"Using specified last 4D layer from metadata: {LAST_CONV}")
                return LAST_CONV
        except Exception:
            print(f"Warning: Specified LAST_CONV '{LAST_CONV}' from metadata not found or not 4D. Falling back to auto-detection.")

    # Fallback to auto-detection
    for layer in reversed(model.layers):
        try:
            out_shape = layer.output_shape
            if isinstance(out_shape, list):
                out_shape = out_shape[0]
            if len(out_shape) == 4:
                print(f"Found last 4D layer via auto-detection: {layer.name}")
                return layer.name
        except Exception:
            continue
    raise ValueError("No 4D conv layer found; set TARGET_LAYER_NAME manually.")


# Redefine make_gradcam_heatmap to use call_like and the updated find_last_conv_layer
def make_gradcam_heatmap(
    x: tf.Tensor,
    class_index: int | None = None,
    last_conv_layer_name: str | None = None,
):
    """Generates a Grad-CAM heatmap for a given image tensor."""
    # Determine the last convolutional layer name
    last_conv_layer_name = last_conv_layer_name or find_last_conv_layer(model)


    conv_layer = model.get_layer(last_conv_layer_name)

    # Build a grad model that preserves the original input structure
    grad_model = Model(inputs=model.inputs, outputs=[conv_layer.output, model.output])

    with tf.GradientTape() as tape:
        tape.watch(x)  # make sure gradients can flow wrt the input image
        # <<< KEY CHANGE: call the grad_model using call_like >>>
        conv_outputs, preds = call_like(grad_model, x)

        if class_index is None:
            # Assuming preds is a tensor (batch_size, num_classes) or similar structure
            # If model output is more complex, this might need adjustment
            if isinstance(preds, (list, tuple)):
                 # Assuming the first element is the primary output
                 preds_tensor = preds[0]
            else:
                 preds_tensor = preds

            if preds_tensor.shape.rank == 2:
                class_index = int(tf.argmax(preds_tensor[0]))
            else:
                 # Handle other potential output shapes if necessary
                 raise ValueError(f"Model output shape not supported for auto class index: {preds_tensor.shape}")


        # Calculate the target score
        target = preds_tensor[:, class_index]


    grads = tape.gradient(target, conv_outputs)               # (1, H, W, C)
    pooled = tf.reduce_mean(grads, axis=(0, 1, 2))            # (C,)
    fmap  = conv_outputs[0]                                   # (H, W, C)
    cam   = tf.reduce_sum(fmap * pooled, axis=-1)             # (H, W)

    cam = tf.maximum(cam, 0.0)
    mx  = tf.reduce_max(cam)
    cam = tf.where(mx > 0, cam / mx, tf.zeros_like(cam))
    return cam.numpy()


# Redefine overlay_heatmap_on_image
def overlay_heatmap_on_image(path: str, heatmap: np.ndarray, alpha=ALPHA, img_size=IMG_SIZE):
    """Overlays a heatmap onto the original image."""
    img = Image.open(path).convert("RGB").resize((img_size, img_size))
    img_np = np.array(img)
    hm = cv2.resize(heatmap, (img_size, img_size))
    hm = np.uint8(255 * hm)
    hm_color = cv2.applyColorMap(hm, cv2.COLORMAP_JET)
    # Ensure dimensions match for cv2.addWeighted
    if img_np.shape[-1] == 3 and hm_color.shape[-1] == 3:
         overlay = cv2.addWeighted(hm_color, alpha, img_np[:, :, ::-1], 1.0 - alpha, 0)
    else:
         # Handle cases where image might be grayscale or heatmap is not 3 channels
         # This simplified version assumes both are RGB or compatible
         print("Warning: Image or heatmap not in expected format for overlay. Skipping overlay.")
         return img_np, img_np # Return original image for both

    overlay = overlay[:, :, ::-1] # Convert BGR to RGB
    return img_np, overlay

# Redefine border_attention_fraction
def border_attention_fraction(heatmap: np.ndarray, border_ratio: float = 0.08) -> float:
    """Calculates the fraction of attention in the border region of a heatmap."""
    h, w = heatmap.shape
    b = int(round(min(h, w) * border_ratio))
    core = heatmap[b:h-b, b:w-b].sum() if (b > 0 and (h - 2 * b) > 0 and (w - 2 * b) > 0) else 0.0
    total = heatmap.sum() + 1e-8
    border = total - core
    return float(border / total)


def pick_samples(score_df: pd.DataFrame, per_class_mis:int, per_class_ok:int):
    """Selects correctly classified and misclassified images per class."""
    rows = []
    for cls in CLASS_NAMES:
        sub = score_df[score_df["true_label"] == cls].copy()
        # Sort by confidence to get the "most" misclassified/correct
        mis = sub[~sub["is_correct"]].sort_values("confidence", ascending=False).head(per_class_mis)
        ok  = sub[sub["is_correct"]].sort_values("confidence", ascending=False).head(per_class_ok)
        rows.append(("MIS", cls, mis))
        rows.append(("OK",  cls, ok))
    return rows


def panel_for_group(kind: str, cls: str, group_df: pd.DataFrame, save_path: Path):
    """Generates and saves a panel of visualizations for a group of images."""
    n = len(group_df)
    if n == 0:
        print(f"No samples for {kind}: {cls}")
        return

    cols = 3 # Original, Heatmap, Overlay
    rows = math.ceil(n) # Each sample gets one row
    fig_h = max(4, rows * 3) # Adjust figure height based on number of rows
    fig_w = 12

    plt.figure(figsize=(fig_w, fig_h))
    idx = 1
    records = []

    for r in group_df.itertuples(index=False):
        try:
            x = preprocess_for_single(r.path)
            heat = make_gradcam_heatmap(x, class_index=r.pred_idx)
            img_np, overlay = overlay_heatmap_on_image(r.path, heat, alpha=ALPHA, img_size=IMG_SIZE)
            frac = border_attention_fraction(heat)

            # Original Image
            plt.subplot(rows, cols, idx);
            plt.imshow(img_np);
            plt.axis("off");
            plt.title(f"Orig\ntrue={r.true_label}\npred={r.pred_label}\nconf={r.confidence:.2f}")
            idx += 1

            # Heatmap
            plt.subplot(rows, cols, idx);
            plt.imshow(heat, cmap="jet");
            plt.axis("off");
            plt.title("Heatmap")
            idx += 1

            # Overlay
            plt.subplot(rows, cols, idx);
            plt.imshow(overlay);
            plt.axis("off");
            plt.title(f"Overlay\nborder={frac:.2f}")
            idx += 1

            records.append({
                "path": r.path,
                "true_label": r.true_label,
                "pred_label": r.pred_label,
                "confidence": r.confidence,
                "border_attention_frac": frac,
                "is_correct": r.is_correct
            })
        except Exception as e:
            print(f"Error processing image {r.path}: {e}")
            # Add a placeholder or skip the row if processing fails
            # For now, let's just print the error and continue

    if records: # Only try to save if there were successful records
        plt.suptitle(f"{kind}: {cls} — {len(records)} samples", y=1.02)
        plt.tight_layout()
        plt.savefig(save_path, dpi=160, bbox_inches="tight")
        plt.show()

        pd.DataFrame(records).to_csv(save_path.with_suffix(".csv"), index=False)
    else:
        print(f"No successful visualizations for {kind}: {cls}. Skipping save.")
    plt.close() # Close the figure to free memory


# Select samples
samples = pick_samples(score_df, N_MISCLASS_PER_CLASS, N_CORRECT_PER_CLASS)

# Generate and save panels
for kind, cls, gdf in samples:
    slug = f"{kind.lower()}_{cls}".replace(" ", "_")
    out_file = OUT_DIR / f"gradcam_{slug}.png"
    panel_for_group(kind, cls, gdf, out_file)

print("Saved panels to:", OUT_DIR.resolve())

Found last 4D layer via auto-detection: resnet50
Error processing image final/val/human/7a89ac6facd28bd8.jpg: Could not call model with structured inputs. Input names=None; got tensor shape=(1, 224, 224, 3). Errors: Tensor call failed ("Exception encountered when calling Functional.call().\n\n\x1b132608572170144\x1b\n\nArguments received by Functional.call():\n  • inputs=tf.Tensor(shape=(1, 224, 224, 3), dtype=float32)\n  • training=False\n  • mask=None\n  • kwargs=<class 'inspect._empty'>"), Dict call failed ("Exception encountered when calling Functional.call().\n\n\x1b132608572170144\x1b\n\nArguments received by Functional.call():\n  • inputs={'image': 'tf.Tensor(shape=(1, 224, 224, 3), dtype=float32)'}\n  • training=False\n  • mask={'image': 'None'}\n  • kwargs=<class 'inspect._empty'>")
Found last 4D layer via auto-detection: resnet50
Error processing image final/val/human/dbef3f001468883a.jpg: Could not call model with structured inputs. Input names=None; got tensor shape=(1, 224

## Visualize grad-cam panels retry 1

### Subtask:
Select samples (correctly classified and misclassified) for each class and generate/save visualizations showing the original image, heatmap, and overlay with border attention score.


## Single-image grad-cam helper

### Subtask:
Single-image grad-cam helper

#### Instructions
Provide a utility function to generate Grad-CAM for a single specified image.


**Reasoning**:
Define the `gradcam_single_image` function as instructed, incorporating the previously defined utility functions and error handling mechanisms.



In [40]:
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf

def gradcam_single_image(path: str, class_idx: int = None):
    """Generates and displays Grad-CAM visualization for a single image."""
    try:
        # Preprocess the image
        x = preprocess_for_single(path)

        # If class_idx is not provided, get the prediction and determine the class index
        if class_idx is None:
            # Use call_like to handle potential input structure variations
            p = call_like(model, x)
            # Assuming the output is a tensor or list/tuple where the first element is the prediction tensor
            p_tensor = p[0] if isinstance(p, (list, tuple)) else p

            if p_tensor.shape.rank == 2 and p_tensor.shape[0] == 1:
                 class_idx = int(np.argmax(p_tensor[0]))
                 predicted_label = index_to_label.get(class_idx, "Unknown")
                 print(f"Predicted class index: {class_idx} ({predicted_label})")
            else:
                 print(f"Warning: Model output shape {p_tensor.shape} not supported for auto class index determination.")
                 # Fallback: attempt to use argmax on the flattened output if it's a single prediction
                 try:
                     class_idx = int(np.argmax(tf.flatten(p_tensor)))
                     predicted_label = index_to_label.get(class_idx, "Unknown")
                     print(f"Attempted auto class index determination via flatten: {class_idx} ({predicted_label})")
                 except Exception as e_flatten:
                     raise ValueError(f"Could not determine class index automatically. Model output shape: {p_tensor.shape}. Error during flatten attempt: {e_flatten}")

        # Generate the heatmap
        heat = make_gradcam_heatmap(x, class_index=class_idx)

        # Overlay the heatmap on the image
        img_np, overlay = overlay_heatmap_on_image(path, heat, alpha=ALPHA, img_size=IMG_SIZE)

        # Calculate border attention fraction
        frac = border_attention_fraction(heat)

        # Create the visualization panel
        plt.figure(figsize=(10,3))
        plt.subplot(1,3,1); plt.imshow(img_np); plt.axis("off"); plt.title("Original")
        plt.subplot(1,3,2); plt.imshow(heat, cmap="jet"); plt.axis("off"); plt.title("Heatmap")
        plt.subplot(1,3,3); plt.imshow(overlay); plt.axis("off"); plt.title(f"Overlay\nborder={frac:.2f}")
        plt.tight_layout()
        plt.show()

    except Exception as e:
        print(f"Error generating Grad-CAM for image {path}: {e}")
        # Optionally display the original image even if Grad-CAM fails
        try:
            img = Image.open(path).convert("RGB").resize((IMG_SIZE, IMG_SIZE))
            plt.figure(figsize=(4,4))
            plt.imshow(np.array(img))
            plt.title(f"Error: {e}\nOriginal Image")
            plt.axis("off")
            plt.show()
        except Exception as img_e:
            print(f"Also failed to display original image: {img_e}")


## Interpretation guidance

### Subtask:
Add markdown explaining how to interpret the Grad-CAM outputs.


**Reasoning**:
Add a markdown cell to explain how to interpret the Grad-CAM outputs, covering correctly classified and misclassified examples, and the border attention score.



In [41]:
# This is a markdown cell to be added to the notebook.
# Add this markdown block between cell '61daf068' and cell 'd667afd2' or as cell 'd667afd2'.
# Remove the existing cell 'd667afd2' if it's just the header "## 10) What to Look For".

"""
## 10) Interpreting Grad‑CAM Outputs

Grad‑CAM visualizations help us understand **which parts of the image the model is focusing on** when making a prediction. The heatmap overlay shows areas of high attention (usually in warmer colors like red/yellow) and low attention (cooler colors like blue).

Here's what to look for:

*   **For Correctly Classified Images:** The heatmap should ideally be concentrated on the main object of interest in the image – for example, the human's face and body, the avatar character, or the animal. This indicates the model is using relevant visual cues for its decision.

*   **For Misclassified Images:** Pay close attention to where the heatmap is focused. If the model is misclassifying an image, its attention might be directed to:
    *   **Backgrounds:** The model might be learning to associate certain backgrounds with a class.
    *   **Borders, Logos, Watermarks, or Text:** Attention on these elements can indicate the model is relying on "shortcut" features rather than the core subject of the image. This is particularly important for profile picture classifiers, where logos or borders might be present.

*   **Border Attention Score:** The "border attention" score provides a quantitative measure of how much of the model's attention (as captured by the heatmap) is focused within a defined border region of the image.
    *   A **high border attention score** suggests the model is significantly influenced by features near the image edges, which could be a sign of relying on problematic shortcuts like logos or watermarks, especially if these features are correlated with specific classes in the training data.
    *   If you observe consistently high border attention scores, particularly for misclassified images, consider strategies like **masking or cropping** these border regions during training or inference, or using **augmentations** that randomize the content or presence of features at the image edges.
"""

'\n## 10) Interpreting Grad‑CAM Outputs\n\nGrad‑CAM visualizations help us understand **which parts of the image the model is focusing on** when making a prediction. The heatmap overlay shows areas of high attention (usually in warmer colors like red/yellow) and low attention (cooler colors like blue).\n\nHere\'s what to look for:\n\n*   **For Correctly Classified Images:** The heatmap should ideally be concentrated on the main object of interest in the image – for example, the human\'s face and body, the avatar character, or the animal. This indicates the model is using relevant visual cues for its decision.\n\n*   **For Misclassified Images:** Pay close attention to where the heatmap is focused. If the model is misclassifying an image, its attention might be directed to:\n    *   **Backgrounds:** The model might be learning to associate certain backgrounds with a class.\n    *   **Borders, Logos, Watermarks, or Text:** Attention on these elements can indicate the model is relying on 

## Summary:

### Data Analysis Key Findings

*   The necessary libraries for the GradCAM analysis were successfully installed.
*   The Keras model and its metadata (class names, image size, input name, input dtype, and last convolutional layer name) were successfully loaded from the specified path.
*   A Pandas DataFrame indexing the images and their labels from the target split ("val") was successfully built, containing 2987 images across "animal", "avatar", and "human" classes.
*   A `tf.data` pipeline for loading and preprocessing images was successfully set up.
*   Inference was successfully run on the dataset, and a score table (`score_df`) containing true labels, predicted labels, confidence scores, and correctness was created.
*   Utility functions for finding the last convolutional layer, generating Grad-CAM heatmaps, overlaying heatmaps on images, and calculating border attention were implemented.
*   Attempts to generate and save Grad-CAM visualizations for selected correctly classified and misclassified samples failed due to a persistent `TypeError` when calling the loaded Keras model within the `tf.GradientTape` context used for heatmap generation.
*   Despite implementing error handling within the visualization loop and attempting different model calling strategies (`call_like` function), the core issue with the model's `call()` method in this specific context could not be resolved.
*   A helper function `gradcam_single_image` was defined for generating visualizations for a single image, incorporating the same utility functions, but it would also be impacted by the model calling error.
*   Markdown guidance on interpreting Grad-CAM outputs and border attention scores was prepared.

### Insights or Next Steps

*   The primary blocker for generating Grad-CAM visualizations is the inability to successfully call the loaded Keras model within the gradient computation context. Further investigation is needed to understand why the model's `call()` method fails with the provided input tensor shape and structure in this specific setup, potentially involving examining the model's internal structure or Keras 3 compatibility issues.
*   If the model calling issue cannot be resolved, alternative approaches for Grad-CAM or other interpretability methods compatible with the loaded model structure might need to be explored.
